Biogrid:
/kaggle/input/datasets/bassam165/biogrid/BIOGRID-ORGANISM-5.0.258.tab3/BIOGRID-ORGANISM-Homo_sapiens-5.0.258.tab3.txt
CTD:
/kaggle/input/datasets/bassam165/comparative-toxicogenomics-database/CTD_curated_genes_diseases.tsv
/kaggle/input/datasets/bassam165/comparative-toxicogenomics-database/CTD_curated_chemicals_diseases.tsv
/kaggle/input/datasets/bassam165/comparative-toxicogenomics-database/CTD_chem_gene_ixns.tsv
/kaggle/input/datasets/bassam165/comparative-toxicogenomics-database/CTD_genes.tsv
/kaggle/input/datasets/bassam165/comparative-toxicogenomics-database/CTD_chemicals.tsv
/kaggle/input/datasets/bassam165/comparative-toxicogenomics-database/CTD_diseases_pathways.tsv
/kaggle/input/datasets/bassam165/comparative-toxicogenomics-database/CTD_genes_pathways.tsv
DGIdb:
/kaggle/input/datasets/bassam165/dgidb-drug-gene-interaction-database/interactions.tsv
/kaggle/input/datasets/bassam165/dgidb-drug-gene-interaction-database/drugs.tsv
/kaggle/input/datasets/bassam165/dgidb-drug-gene-interaction-database/categories.tsv

In [1]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "torch==2.4.1", "torchvision==0.19.1",
                "--index-url", "https://download.pytorch.org/whl/cu121"],
               check=True)
print("done — now restart the kernel (Run > Restart) and run from Step 1")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.9/798.9 MB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 111.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 76.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 42.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 99.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 7.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 33.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 8.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 9.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.2/176.2 MB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchaudio 2.10.0+cu128 requires torch==2.10.0, but you have torch 2.4.1+cu121 which is incompatible.


done — now restart the kernel (Run > Restart) and run from Step 1


In [2]:
!pip install -q node2vec --no-deps
!pip install -q gensim

# Step 1 — Setup, loading, cleaning, ID canonicalization

In [3]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "torch==2.4.1", "torchvision==0.19.1",
                "--index-url", "https://download.pytorch.org/whl/cu121"],
               check=True)

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import gc
import pickle
import time
import warnings
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy import sparse
from scipy.stats import wilcoxon, shapiro, ttest_rel, fisher_exact, hypergeom
from statsmodels.stats.multitest import multipletests
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, matthews_corrcoef
from sklearn.linear_model import LogisticRegression
warnings.filterwarnings("ignore")

BIOGRID_PATH = "/kaggle/input/datasets/bassam165/biogrid/BIOGRID-ORGANISM-5.0.258.tab3/BIOGRID-ORGANISM-Homo_sapiens-5.0.258.tab3.txt"
CTD_DIR = "/kaggle/input/datasets/bassam165/comparative-toxicogenomics-database"
DGIDB_DIR = "/kaggle/input/datasets/bassam165/dgidb-drug-gene-interaction-database"

DISEASE_ID = "MESH:D005909"
DISEASE_NAME = "Glioblastoma"
HUMAN_TAXID = 9606

OUT_DIR = "/kaggle/working"
FIG_DIR = os.path.join(OUT_DIR, "figures")
MODEL_DIR = os.path.join(OUT_DIR, "models")
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
print("torch:", torch.__version__, "| device:", device, "| arch:", torch.cuda.get_arch_list() if torch.cuda.is_available() else "cpu")

BIOGRID_COLS = [
    "biogrid_interaction_id", "entrez_a", "entrez_b", "biogrid_a", "biogrid_b",
    "systematic_a", "systematic_b", "symbol_a", "symbol_b", "synonyms_a", "synonyms_b",
    "exp_system", "exp_system_type", "author", "pubmed", "organism_a", "organism_b",
    "throughput", "score", "modification", "qualifications", "tags", "source_db",
    "swissprot_a", "trembl_a", "trembl_b2", "refseq_a", "swissprot_b", "trembl_b",
    "swissprot_b2", "refseq_b", "ph_a", "ph_b", "ph_c", "ph_d", "org_name_a", "org_name_b"
]
biogrid = pd.read_csv(BIOGRID_PATH, sep="\t", header=None, names=BIOGRID_COLS, dtype=str, low_memory=False)
biogrid = biogrid[(biogrid["organism_a"] == str(HUMAN_TAXID)) & (biogrid["organism_b"] == str(HUMAN_TAXID))]
biogrid = biogrid[["entrez_a", "entrez_b"]].dropna()
biogrid = biogrid[(biogrid["entrez_a"] != "-") & (biogrid["entrez_b"] != "-")]
biogrid = biogrid[biogrid["entrez_a"].str.isdigit() & biogrid["entrez_b"].str.isdigit()]
biogrid["entrez_a"] = biogrid["entrez_a"].astype(int)
biogrid["entrez_b"] = biogrid["entrez_b"].astype(int)
biogrid = biogrid[biogrid["entrez_a"] != biogrid["entrez_b"]]
biogrid["pair"] = biogrid.apply(lambda r: tuple(sorted((r["entrez_a"], r["entrez_b"]))), axis=1)
biogrid = biogrid.drop_duplicates("pair").drop(columns="pair").reset_index(drop=True)
print("BioGRID human PPI edges:", len(biogrid))
print("Unique genes in PPI:", len(set(biogrid["entrez_a"]) | set(biogrid["entrez_b"])))

GD_COLS = ["gene_symbol", "gene_id", "disease_name", "disease_id", "direct_evidence", "omim_ids", "pubmed_ids"]
ctd_gd = pd.read_csv(os.path.join(CTD_DIR, "CTD_curated_genes_diseases.tsv"), sep="\t", header=None, names=GD_COLS, dtype=str, low_memory=False)
ctd_gd = ctd_gd.dropna(subset=["gene_id", "disease_id"])
ctd_gd = ctd_gd[ctd_gd["gene_id"].str.isdigit()]
ctd_gd["gene_id"] = ctd_gd["gene_id"].astype(int)

CD_COLS = ["chemical_name", "chemical_id", "cas_rn", "disease_name", "disease_id", "direct_evidence", "pubmed_ids"]
ctd_cd = pd.read_csv(os.path.join(CTD_DIR, "CTD_curated_chemicals_diseases.tsv"), sep="\t", header=None, names=CD_COLS, dtype=str, low_memory=False)
ctd_cd = ctd_cd.dropna(subset=["chemical_id", "disease_id"])

CGI_COLS = ["chemical_name", "chemical_id", "cas_rn", "gene_symbol", "gene_id", "gene_forms", "organism", "organism_id", "interaction", "interaction_actions", "pubmed_ids"]
ctd_cgi = pd.read_csv(os.path.join(CTD_DIR, "CTD_chem_gene_ixns.tsv"), sep="\t", header=None, names=CGI_COLS, dtype=str, low_memory=False)
ctd_cgi = ctd_cgi[ctd_cgi["organism_id"] == str(HUMAN_TAXID)]
ctd_cgi = ctd_cgi.dropna(subset=["chemical_id", "gene_id"])
ctd_cgi = ctd_cgi[ctd_cgi["gene_id"].str.isdigit()]
ctd_cgi["gene_id"] = ctd_cgi["gene_id"].astype(int)

GENES_COLS = ["gene_symbol", "gene_name", "gene_id", "alt_gene_ids", "synonyms", "biogrid_ids", "pharmgkb_ids", "uniprot_ids"]
ctd_genes = pd.read_csv(os.path.join(CTD_DIR, "CTD_genes.tsv"), sep="\t", header=None, names=GENES_COLS, dtype=str, low_memory=False)
ctd_genes = ctd_genes.dropna(subset=["gene_id"])
ctd_genes = ctd_genes[ctd_genes["gene_id"].str.isdigit()]
ctd_genes["gene_id"] = ctd_genes["gene_id"].astype(int)

CHEM_COLS = ["chemical_name", "chemical_id", "cas_rn", "pubchem_cid", "pubchem_sid", "dtxsid", "inchikey", "definition", "parent_ids", "tree_numbers", "parent_tree_numbers", "mesh_synonyms", "curated_synonyms"]
ctd_chem = pd.read_csv(os.path.join(CTD_DIR, "CTD_chemicals.tsv"), sep="\t", header=None, names=CHEM_COLS, dtype=str, low_memory=False)
ctd_chem = ctd_chem.dropna(subset=["chemical_id"])
ctd_chem["chemical_id"] = ctd_chem["chemical_id"].str.replace("MESH:", "", regex=False)
print("CTD gene-disease:", len(ctd_gd), "| chem-disease:", len(ctd_cd), "| chem-gene:", len(ctd_cgi))
print("CTD genes vocab:", len(ctd_genes), "| chemicals vocab:", len(ctd_chem))

DGI_INT_COLS = ["gene_claim_name", "gene_concept_id", "gene_name", "source_db_name", "source_db_version", "interaction_type", "interaction_score", "drug_claim_name", "drug_concept_id", "drug_name", "approved", "immunotherapy", "anti_neoplastic"]
dgidb = pd.read_csv(os.path.join(DGIDB_DIR, "interactions.tsv"), sep="\t", header=None, names=DGI_INT_COLS, dtype=str, low_memory=False, skiprows=1)
dgidb = dgidb.dropna(subset=["gene_name", "drug_name"])

DGI_DRUGS_COLS = ["drug_claim_name", "nomenclature", "concept_id", "drug_name", "approved", "immunotherapy", "anti_neoplastic", "source_db_name", "source_db_version"]
dgidb_drugs = pd.read_csv(os.path.join(DGIDB_DIR, "drugs.tsv"), sep="\t", header=None, names=DGI_DRUGS_COLS, dtype=str, low_memory=False, skiprows=1)

DGI_CAT_COLS = ["gene_name", "category", "source_db_name", "source_db_version"]
dgidb_cat = pd.read_csv(os.path.join(DGIDB_DIR, "categories.tsv"), sep="\t", header=None, names=DGI_CAT_COLS, dtype=str, low_memory=False, skiprows=1)
print("DGIdb interactions:", len(dgidb), "| drugs:", len(dgidb_drugs), "| categories:", len(dgidb_cat))

sym2entrez = {}
for _, row in ctd_genes.iterrows():
    sym = row["gene_symbol"]
    if isinstance(sym, str):
        sym2entrez[sym.upper()] = row["gene_id"]
    if isinstance(row["synonyms"], str):
        for s in row["synonyms"].split("|"):
            sym2entrez.setdefault(s.upper(), row["gene_id"])

dgidb["gene_upper"] = dgidb["gene_name"].astype(str).str.upper()
dgidb["gene_id"] = dgidb["gene_upper"].map(sym2entrez)
dgidb_mapped = dgidb.dropna(subset=["gene_id"]).copy()
dgidb_mapped["gene_id"] = dgidb_mapped["gene_id"].astype(int)
print("DGIdb rows mapped to Entrez:", len(dgidb_mapped), "/", len(dgidb), f"({100*len(dgidb_mapped)/len(dgidb):.1f}%)")

chem_name2mesh = {}
for _, row in ctd_chem.iterrows():
    cname = row["chemical_name"]
    if isinstance(cname, str):
        chem_name2mesh[cname.upper()] = row["chemical_id"]
    if isinstance(row["mesh_synonyms"], str):
        for s in row["mesh_synonyms"].split("|"):
            chem_name2mesh.setdefault(s.upper(), row["chemical_id"])
dgidb_mapped["drug_upper"] = dgidb_mapped["drug_name"].astype(str).str.upper()
dgidb_mapped["chemical_id"] = dgidb_mapped["drug_upper"].map(chem_name2mesh)
print("DGIdb drug rows mapped to MeSH ChemicalID:", dgidb_mapped["chemical_id"].notna().sum())

seed_df = ctd_gd[ctd_gd["disease_id"] == DISEASE_ID].copy()
seed_genes = sorted(set(seed_df["gene_id"]))
ppi_genes = set(biogrid["entrez_a"]) | set(biogrid["entrez_b"])
seed_in_ppi = [g for g in seed_genes if g in ppi_genes]
print(f"{DISEASE_NAME} seed genes (curated):", len(seed_genes), "| in PPI:", len(seed_in_ppi))

pos_df = ctd_cd[(ctd_cd["disease_id"] == DISEASE_ID) & (ctd_cd["direct_evidence"].fillna("").str.contains("therapeutic"))].copy()
pos_drugs = sorted(set(pos_df["chemical_id"].str.replace("MESH:", "", regex=False)))
print(f"{DISEASE_NAME} therapeutic positive drugs (case-study validation set):", len(pos_drugs))

artifacts = {
    "biogrid": biogrid, "ctd_gd": ctd_gd, "ctd_cd": ctd_cd, "ctd_cgi": ctd_cgi,
    "ctd_genes": ctd_genes, "ctd_chem": ctd_chem, "dgidb_mapped": dgidb_mapped, "dgidb_cat": dgidb_cat,
    "sym2entrez": sym2entrez, "chem_name2mesh": chem_name2mesh,
    "seed_genes": seed_genes, "seed_in_ppi": seed_in_ppi, "pos_drugs": pos_drugs, "ppi_genes": ppi_genes,
}
with open(os.path.join(OUT_DIR, "step1_artifacts.pkl"), "wb") as f:
    pickle.dump(artifacts, f)
print("Step 1 artifacts saved.")
gc.collect()

torch: 2.4.1+cu121 | device: cuda | arch: ['sm_50', 'sm_60', 'sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90']
BioGRID human PPI edges: 968909
Unique genes in PPI: 20487
CTD gene-disease: 34253 | chem-disease: 109362 | chem-gene: 1321614
CTD genes vocab: 642392 | chemicals vocab: 179481
DGIdb interactions: 81314 | drugs: 81572 | categories: 32795
DGIdb rows mapped to Entrez: 80519 / 81314 (99.0%)
DGIdb drug rows mapped to MeSH ChemicalID: 53263
Glioblastoma seed genes (curated): 79 | in PPI: 79
Glioblastoma therapeutic positive drugs (case-study validation set): 80
Step 1 artifacts saved.


0

# Step 2 — Graph construction, RWR disease module, node features

In [4]:
with open(os.path.join(OUT_DIR, "step1_artifacts.pkl"), "rb") as f:
    A = pickle.load(f)

biogrid = A["biogrid"]; ctd_cgi = A["ctd_cgi"]; ctd_chem = A["ctd_chem"]
dgidb_mapped = A["dgidb_mapped"]; dgidb_cat = A["dgidb_cat"]; ctd_genes = A["ctd_genes"]
seed_genes = A["seed_genes"]; pos_drugs = A["pos_drugs"]

G_ppi = nx.Graph()
G_ppi.add_edges_from(zip(biogrid["entrez_a"], biogrid["entrez_b"]))
G_ppi.remove_edges_from(nx.selfloop_edges(G_ppi))
lcc_nodes = max(nx.connected_components(G_ppi), key=len)
G_ppi = G_ppi.subgraph(lcc_nodes).copy()
gene_nodes = sorted(G_ppi.nodes())
gene_idx = {g: i for i, g in enumerate(gene_nodes)}
n_genes = len(gene_nodes)
print("PPI LCC genes:", n_genes, "| edges:", G_ppi.number_of_edges())

seed_in_lcc = [g for g in seed_genes if g in gene_idx]
print("Seed genes in LCC:", len(seed_in_lcc), "/", len(seed_genes))

cgi_edges = ctd_cgi[ctd_cgi["gene_id"].isin(gene_idx)][["chemical_id", "gene_id"]].copy()
cgi_edges["chemical_id"] = cgi_edges["chemical_id"].str.replace("MESH:", "", regex=False)
cgi_edges = cgi_edges.rename(columns={"chemical_id": "drug"})

dgi_edges = dgidb_mapped.dropna(subset=["chemical_id"])[["chemical_id", "gene_id"]].copy()
dgi_edges["chemical_id"] = dgi_edges["chemical_id"].str.replace("MESH:", "", regex=False)
dgi_edges = dgi_edges[dgi_edges["gene_id"].isin(gene_idx)].rename(columns={"chemical_id": "drug"})

drug_gene = pd.concat([cgi_edges, dgi_edges], ignore_index=True).drop_duplicates().reset_index(drop=True)
drug_gene["gene_id"] = drug_gene["gene_id"].astype(int)
print("Drug-gene edges (union, in LCC):", len(drug_gene))

drug_counts = drug_gene["drug"].value_counts()
valid_drugs = set(drug_counts[drug_counts >= 1].index)
drug_nodes = sorted(valid_drugs)
drug_idx = {d: i for i, d in enumerate(drug_nodes)}
n_drugs = len(drug_nodes)
drug_gene = drug_gene[drug_gene["drug"].isin(drug_idx)].reset_index(drop=True)
print("Drug nodes:", n_drugs)

pos_drugs_in = [d for d in pos_drugs if d in drug_idx]
print(f"{DISEASE_NAME} therapeutic drugs present as nodes:", len(pos_drugs_in))

adj = nx.to_scipy_sparse_array(G_ppi, nodelist=gene_nodes, format="csr", dtype=np.float64)
deg = np.asarray(adj.sum(axis=1)).flatten(); deg[deg == 0] = 1.0
P = adj.multiply(1.0 / deg[:, None]).tocsr()

restart = 0.3
seed_vec = np.zeros(n_genes)
for g in seed_in_lcc:
    seed_vec[gene_idx[g]] = 1.0
seed_vec /= seed_vec.sum()
rwr = seed_vec.copy()
for _ in range(100):
    rwr_next = (1 - restart) * P.T.dot(rwr) + restart * seed_vec
    if np.linalg.norm(rwr_next - rwr, 1) < 1e-10:
        rwr = rwr_next; break
    rwr = rwr_next
rwr = rwr / rwr.max()
print("RWR proximity computed. Nonzero:", int((rwr > 1e-8).sum()))

pr = nx.pagerank(G_ppi, alpha=0.85)
clust = nx.clustering(G_ppi)
core = nx.core_number(G_ppi)
betw = nx.betweenness_centrality(G_ppi, k=min(500, n_genes), seed=SEED)

gene_feat = np.zeros((n_genes, 6), dtype=np.float32)
seed_set = set(seed_in_lcc)
for g in gene_nodes:
    i = gene_idx[g]
    gene_feat[i, 0] = G_ppi.degree(g)
    gene_feat[i, 1] = betw[g]
    gene_feat[i, 2] = clust[g]
    gene_feat[i, 3] = core[g]
    gene_feat[i, 4] = pr[g]
    gene_feat[i, 5] = 1.0 if g in seed_set else 0.0
gene_feat[:, 0] = np.log1p(gene_feat[:, 0])
for c in range(5):
    col = gene_feat[:, c]; std = col.std()
    gene_feat[:, c] = (col - col.mean()) / std if std > 0 else col
rwr_feat = (rwr - rwr.mean()) / (rwr.std() + 1e-12)
gene_feat = np.hstack([gene_feat, rwr_feat.reshape(-1, 1).astype(np.float32)])
print("Gene feature matrix:", gene_feat.shape)

def hash_fp(n, dim=128):
    rng = np.random.RandomState(abs(hash(n)) % (2**31))
    return rng.randint(0, 2, dim).astype(np.float32)

FP_DIM = 128
drug_fp = np.zeros((n_drugs, FP_DIM), dtype=np.float32)
for d in drug_nodes:
    drug_fp[drug_idx[d]] = hash_fp(d, FP_DIM)

cat_map = {}
for _, row in dgidb_cat.iterrows():
    if isinstance(row["gene_name"], str) and isinstance(row["category"], str):
        cat_map.setdefault(row["gene_name"].upper(), set()).add(row["category"])
top_cats = pd.Series([c for s in cat_map.values() for c in s]).value_counts().head(20).index.tolist()
cat_idx = {c: i for i, c in enumerate(top_cats)}
sym_of_gene = ctd_genes.set_index("gene_id")["gene_symbol"].to_dict()
drug_targets = drug_gene.groupby("drug")["gene_id"].apply(list).to_dict()
drug_cat = np.zeros((n_drugs, len(top_cats)), dtype=np.float32)
for d in drug_nodes:
    for g in drug_targets.get(d, []):
        sym = str(sym_of_gene.get(g, "")).upper()
        for c in cat_map.get(sym, []):
            if c in cat_idx:
                drug_cat[drug_idx[d], cat_idx[c]] = 1.0
drug_feat = np.hstack([drug_fp, drug_cat])
print("Drug feature matrix:", drug_feat.shape)

gg_edge = np.array([[gene_idx[a], gene_idx[b]] for a, b in G_ppi.edges()], dtype=np.int64).T
dg_edge = np.array([[drug_idx[d], gene_idx[g]] for d, g in zip(drug_gene["drug"], drug_gene["gene_id"])], dtype=np.int64).T
seed_mask = np.zeros(n_genes, dtype=bool)
for g in seed_in_lcc:
    seed_mask[gene_idx[g]] = True

PROX_THRESH = np.quantile(rwr, 0.90)
gene_proximal = rwr >= PROX_THRESH
print("Disease-proximal genes (top 10% RWR):", int(gene_proximal.sum()))
edge_is_proximal = gene_proximal[dg_edge[1]]
print("Drug-gene edges in disease-proximal slice:", int(edge_is_proximal.sum()), "/", dg_edge.shape[1])

graph = {
    "gene_nodes": gene_nodes, "gene_idx": gene_idx, "n_genes": n_genes,
    "drug_nodes": drug_nodes, "drug_idx": drug_idx, "n_drugs": n_drugs,
    "gene_feat": gene_feat, "drug_feat": drug_feat,
    "gg_edge": gg_edge, "dg_edge": dg_edge,
    "rwr": rwr, "seed_mask": seed_mask, "seed_in_lcc": seed_in_lcc,
    "gene_proximal": gene_proximal, "edge_is_proximal": edge_is_proximal, "PROX_THRESH": float(PROX_THRESH),
    "drug_targets": drug_targets, "pos_drugs_in": pos_drugs_in, "sym_of_gene": sym_of_gene,
}
with open(os.path.join(OUT_DIR, "step2_graph.pkl"), "wb") as f:
    pickle.dump(graph, f)
print("Step 2 graph saved.")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
degs = [d for _, d in G_ppi.degree()]
axes[0].hist(np.log1p(degs), bins=60, color="#3b6ea5", edgecolor="white")
axes[0].set_xlabel("log(1 + degree)"); axes[0].set_ylabel("Gene count"); axes[0].set_title("PPI Backbone Degree Distribution")
order = np.argsort(rwr)[::-1]
axes[1].plot(np.arange(n_genes), rwr[order], color="#a53b5a", lw=1.5)
axes[1].axhline(PROX_THRESH, ls="--", color="gray", lw=1, label="proximal threshold (top 10%)")
axes[1].set_xlabel("Genes ranked by RWR proximity"); axes[1].set_ylabel("RWR heat (normalized)")
axes[1].set_title(f"Disease-Module Proximity ({DISEASE_NAME})"); axes[1].legend()
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "Figure_2_ppi_degree_rwr.png"), dpi=300, bbox_inches="tight")
plt.close(fig); print("Figure 2 saved.")

top_module = [gene_nodes[i] for i in np.argsort(rwr)[::-1][:80]]
H = G_ppi.subgraph(top_module).copy()
pos = nx.spring_layout(H, seed=SEED, k=0.3)
node_color = ["#d62728" if seed_mask[gene_idx[g]] else "#1f77b4" for g in H.nodes()]
node_size = [120 + 400 * rwr[gene_idx[g]] for g in H.nodes()]
fig, ax = plt.subplots(figsize=(11, 9))
nx.draw_networkx_edges(H, pos, alpha=0.2, ax=ax)
nx.draw_networkx_nodes(H, pos, node_color=node_color, node_size=node_size, ax=ax)
labels = {g: sym_of_gene.get(g, str(g)) for g in top_module if seed_mask[gene_idx[g]]}
nx.draw_networkx_labels(H, pos, labels=labels, font_size=7, ax=ax)
ax.set_title(f"{DISEASE_NAME} Disease-Module Subnetwork (top-80 RWR; red = seeds)"); ax.axis("off")
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "Figure_3_disease_module_subnetwork.png"), dpi=300, bbox_inches="tight")
plt.close(fig); print("Figure 3 saved.")
gc.collect()

PPI LCC genes: 20487 | edges: 968909
Seed genes in LCC: 79 / 79
Drug-gene edges (union, in LCC): 826455
Drug nodes: 14904
Glioblastoma therapeutic drugs present as nodes: 76
RWR proximity computed. Nonzero: 20487
Gene feature matrix: (20487, 7)
Drug feature matrix: (14904, 148)
Disease-proximal genes (top 10% RWR): 2049
Drug-gene edges in disease-proximal slice: 168903 / 826455
Step 2 graph saved.
Figure 2 saved.
Figure 3 saved.


27

# Step 3 — Task definition: therapeutic positives, degree-matched negatives, transductive + cold-start drug splits

In [5]:
with open(os.path.join(OUT_DIR, "step2_graph.pkl"), "rb") as f:
    graph = pickle.load(f)

n_genes = graph["n_genes"]; n_drugs = graph["n_drugs"]
dg_edge = graph["dg_edge"]
gene_proximal = graph["gene_proximal"]

pos_edges = np.unique(dg_edge.T.astype(np.int64), axis=0)
n_pos = len(pos_edges)
print("Unique drug-gene positive edges:", n_pos)

pos_set = set((int(d), int(g)) for d, g in pos_edges)

gene_deg = np.bincount(pos_edges[:, 1], minlength=n_genes).astype(np.int64)
n_bins = 25
log_gd = np.log1p(gene_deg)
nz = gene_deg > 0
edges_q = np.quantile(log_gd[nz], np.linspace(0, 1, n_bins + 1))
edges_q[-1] += 1e-6
gene_bin = np.digitize(log_gd, edges_q[1:-1])
genes_in_bin = {b: np.where((gene_bin == b) & nz)[0] for b in range(n_bins)}
for b in range(n_bins):
    if len(genes_in_bin[b]) == 0:
        genes_in_bin[b] = np.where(nz)[0]

rng = np.random.RandomState(SEED)
NEG_RATIO = 1
target_neg = NEG_RATIO * n_pos
neg_set = set()
batch = max(target_neg * 2, 1000000)
while len(neg_set) < target_neg:
    drugs_s = rng.randint(0, n_drugs, batch)
    src_genes = pos_edges[rng.randint(0, n_pos, batch), 1]
    bins = gene_bin[src_genes]
    genes_s = np.empty(batch, dtype=np.int64)
    for b in np.unique(bins):
        m = bins == b
        pool = genes_in_bin[b]
        genes_s[m] = pool[rng.randint(0, len(pool), m.sum())]
    for d, g in zip(drugs_s, genes_s):
        e = (int(d), int(g))
        if e not in pos_set:
            neg_set.add(e)
            if len(neg_set) >= target_neg:
                break
neg_edges = np.array(sorted(neg_set), dtype=np.int64)
print("Degree-matched negative edges:", len(neg_edges))

uni_drugs = rng.randint(0, n_drugs, target_neg * 2)
uni_genes = rng.randint(0, n_genes, target_neg * 2)
uni_neg = []
for d, g in zip(uni_drugs, uni_genes):
    e = (int(d), int(g))
    if e not in pos_set:
        uni_neg.append(e)
        if len(uni_neg) >= target_neg:
            break
uniform_neg = np.array(uni_neg, dtype=np.int64)
print("Uniform-random negative edges (ablation):", len(uniform_neg))

def edge_proximal_mask(edges):
    return gene_proximal[edges[:, 1]]

def split_edges(pos, neg, seed):
    r = np.random.RandomState(seed)
    pi = r.permutation(len(pos)); ni = r.permutation(len(neg))
    def s3(idx, n):
        tr = int(0.8 * n); va = int(0.9 * n)
        return idx[:tr], idx[tr:va], idx[va:]
    ptr, pva, pte = s3(pi, len(pos)); ntr, nva, nte = s3(ni, len(neg))
    def pack(pidx, nidx):
        E = np.vstack([pos[pidx], neg[nidx]])
        y = np.concatenate([np.ones(len(pidx)), np.zeros(len(nidx))]).astype(np.float32)
        return E, y
    return pack(ptr, ntr), pack(pva, nva), pack(pte, nte)

transductive = split_edges(pos_edges, neg_edges, SEED)
(trE, trY), (vaE, vaY), (teE, teY) = transductive
te_prox = edge_proximal_mask(teE)
print("Transductive | train:", len(trY), "val:", len(vaY), "test:", len(teY))
print("Test edges proximal:", int(te_prox.sum()), "| non-proximal:", int((~te_prox).sum()))

r = np.random.RandomState(SEED + 1)
drug_perm = r.permutation(n_drugs)
held_drugs = set(drug_perm[:int(0.2 * n_drugs)].tolist())
def in_held(edges):
    return np.array([int(d) in held_drugs for d in edges[:, 0]])
pos_held = in_held(pos_edges); neg_held = in_held(neg_edges)
cold_train_pos = pos_edges[~pos_held]; cold_test_pos = pos_edges[pos_held]
cold_train_neg = neg_edges[~neg_held]; cold_test_neg = neg_edges[neg_held]
coldE_tr = np.vstack([cold_train_pos, cold_train_neg])
coldY_tr = np.concatenate([np.ones(len(cold_train_pos)), np.zeros(len(cold_train_neg))]).astype(np.float32)
coldE_te = np.vstack([cold_test_pos, cold_test_neg])
coldY_te = np.concatenate([np.ones(len(cold_test_pos)), np.zeros(len(cold_test_neg))]).astype(np.float32)
cold_te_prox = edge_proximal_mask(coldE_te)
print("Cold-start | train edges:", len(coldY_tr), "test edges:", len(coldY_te), "(held-out drugs:", len(held_drugs), ")")

task = {
    "pos_edges": pos_edges, "neg_edges": neg_edges, "uniform_neg": uniform_neg,
    "transductive": transductive, "te_prox": te_prox,
    "cold": {"trE": coldE_tr, "trY": coldY_tr, "teE": coldE_te, "teY": coldY_te,
             "te_prox": cold_te_prox, "held_drugs": np.array(sorted(held_drugs), dtype=np.int64)},
    "gene_bin": gene_bin, "genes_in_bin": {int(k): v for k, v in genes_in_bin.items()},
    "NEG_RATIO": NEG_RATIO, "n_bins": n_bins,
}
with open(os.path.join(OUT_DIR, "step3_task.pkl"), "wb") as f:
    pickle.dump(task, f)
print("Step 3 task saved.")
gc.collect()

Unique drug-gene positive edges: 826455
Degree-matched negative edges: 826455
Uniform-random negative edges (ablation): 826455
Transductive | train: 1322328 val: 165290 test: 165292
Test edges proximal: 32431 | non-proximal: 132861
Cold-start | train edges: 1320044 test edges: 332866 (held-out drugs: 2980 )
Step 3 task saved.


0

# tep 4 — Models: DMAA-HGT backbone (RWR-biased message passing) + DistMult decoder, plus all baselines (GCN, GAT, vanilla HGT, node2vec+LR helper)

In [6]:
with open(os.path.join(OUT_DIR, "step2_graph.pkl"), "rb") as f:
    graph = pickle.load(f)

n_genes = graph["n_genes"]; n_drugs = graph["n_drugs"]
gene_feat = torch.tensor(graph["gene_feat"], dtype=torch.float32, device=device)
drug_feat = torch.tensor(graph["drug_feat"], dtype=torch.float32, device=device)
gg_edge = torch.tensor(graph["gg_edge"], dtype=torch.long, device=device)
dg_edge = torch.tensor(graph["dg_edge"], dtype=torch.long, device=device)
rwr_np = graph["rwr"]
rwr_t = torch.tensor(rwr_np, dtype=torch.float32, device=device)
gg_edge_np = graph["gg_edge"]

def make_bidir(e):
    return torch.cat([e, e.flip(0)], dim=1)
gg_bi = make_bidir(gg_edge)

dg_du = dg_edge[0]
dg_gv = dg_edge[1]

drug_idx = graph["drug_idx"]; gene_idx = graph["gene_idx"]; drug_targets = graph["drug_targets"]
drug_prox_np = np.zeros(n_drugs, dtype=np.float32)
for d, tgts in drug_targets.items():
    if d in drug_idx:
        vals = [rwr_np[gene_idx[g]] for g in tgts if g in gene_idx]
        if vals:
            drug_prox_np[drug_idx[d]] = float(np.mean(vals))
drug_prox_t = torch.tensor(drug_prox_np, dtype=torch.float32, device=device)
gene_prox_t = rwr_t
print("Drug proximity computed. Nonzero:", int((drug_prox_np > 1e-8).sum()), "/", n_drugs)

def deg_inv(index, num_nodes):
    d = torch.zeros(num_nodes, device=index.device).scatter_add_(0, index, torch.ones(index.size(0), device=index.device))
    di = d.pow(-1.0); di[torch.isinf(di)] = 0.0
    return di

def aggregate(x_src, src_idx, dst_idx, num_dst, dim):
    out = torch.zeros(num_dst, dim, device=x_src.device).scatter_add_(0, dst_idx.unsqueeze(-1).expand(-1, dim), x_src[src_idx])
    return out * deg_inv(dst_idx, num_dst).unsqueeze(-1)

class GatedGCN(nn.Module):
    def __init__(self, gene_in, drug_in, dim=64, n_layers=2, dropout=0.4, use_gate=True):
        super().__init__()
        self.use_gate = use_gate
        self.gene_enc = nn.Linear(gene_in, dim); self.drug_enc = nn.Linear(drug_in, dim)
        self.g_lins = nn.ModuleList([nn.Linear(dim, dim) for _ in range(n_layers)])
        self.d_lins = nn.ModuleList([nn.Linear(dim, dim) for _ in range(n_layers)])
        self.dropout = nn.Dropout(dropout)
        self.rel = nn.Parameter(torch.randn(dim) * 0.1)
        self.gene_gate_w = nn.Parameter(torch.tensor(1.0)); self.gene_gate_b = nn.Parameter(torch.tensor(0.0))
        self.drug_gate_w = nn.Parameter(torch.tensor(1.0)); self.drug_gate_b = nn.Parameter(torch.tensor(0.0))
        self.beta_gene = nn.Parameter(torch.tensor(1.0)); self.beta_drug = nn.Parameter(torch.tensor(1.0))

    def gene_gate(self, prox):
        if not self.use_gate: return 1.0
        return 1.0 + self.beta_gene * torch.sigmoid(self.gene_gate_w * prox + self.gene_gate_b).unsqueeze(-1)

    def drug_gate(self, prox):
        if not self.use_gate: return 1.0
        return 1.0 + self.beta_drug * torch.sigmoid(self.drug_gate_w * prox + self.drug_gate_b).unsqueeze(-1)

    def encode(self, gene_x0, drug_x0, gg, du, gv, gene_prox, drug_prox):
        g = self.dropout(F.relu(self.gene_enc(gene_x0)))
        d = self.dropout(F.relu(self.drug_enc(drug_x0)))
        gg_gate = self.gene_gate(gene_prox); dd_gate = self.drug_gate(drug_prox)
        n_g = g.size(0); n_d = d.size(0); dim = g.size(1)
        gg_src, gg_dst = gg[0], gg[1]
        for li in range(len(self.g_lins)):
            g_self = aggregate(g, gg_src, gg_dst, n_g, dim)
            g_from_d = aggregate(d, du, gv, n_g, dim)
            d_from_g = aggregate(g, gv, du, n_d, dim)
            g_new = F.relu(gg_gate * self.g_lins[li](g + g_self + g_from_d))
            d_new = F.relu(dd_gate * self.d_lins[li](d + d_from_g))
            g, d = self.dropout(g_new), self.dropout(d_new)
        return g, d

    def score_edges(self, drug_emb, gene_emb, edges):
        return (drug_emb[edges[:, 0]] * self.rel * gene_emb[edges[:, 1]]).sum(-1)

class HomoGAT(nn.Module):
    def __init__(self, gene_in, drug_in, dim=64, n_layers=2, n_heads=4, dropout=0.4):
        super().__init__()
        self.gene_enc = nn.Linear(gene_in, dim); self.drug_enc = nn.Linear(drug_in, dim)
        self.att = nn.ModuleList([nn.Linear(2 * dim, n_heads) for _ in range(n_layers)])
        self.lins = nn.ModuleList([nn.Linear(dim, dim) for _ in range(n_layers)])
        self.h = n_heads; self.dropout = nn.Dropout(dropout); self.edge_chunk = 200000
        self.rel = nn.Parameter(torch.randn(dim) * 0.1)

    def _gat(self, x, edge_index, num_nodes, att, lin):
        src, dst = edge_index[0], edge_index[1]; E = src.size(0)
        denom = torch.zeros(num_nodes, device=x.device); agg = torch.zeros(num_nodes, x.size(1), device=x.device)
        with torch.no_grad():
            maxl = torch.full((num_nodes,), -1e30, device=x.device)
            for i in range(0, E, self.edge_chunk):
                s = src[i:i+self.edge_chunk]; dd = dst[i:i+self.edge_chunk]
                sc = F.leaky_relu(att(torch.cat([x[s], x[dd]], -1))).mean(-1)
                maxl = maxl.index_reduce(0, dd, sc, "amax", include_self=True)
        for i in range(0, E, self.edge_chunk):
            s = src[i:i+self.edge_chunk]; dd = dst[i:i+self.edge_chunk]
            sc = F.leaky_relu(att(torch.cat([x[s], x[dd]], -1))).mean(-1)
            ex = torch.exp(sc - maxl[dd])
            denom = denom.index_add(0, dd, ex); agg = agg.index_add(0, dd, x[s] * ex.unsqueeze(-1))
        return F.relu(lin(x + agg / (denom.unsqueeze(-1) + 1e-16)))

    def encode(self, gene_x0, drug_x0, gg, du, gv, gene_prox, drug_prox):
        g = F.relu(self.gene_enc(gene_x0)); d = F.relu(self.drug_enc(drug_x0)); n_g = g.size(0); n_d = d.size(0); dim = g.size(1)
        for att, lin in zip(self.att, self.lins):
            g = self.dropout(self._gat(g, gg, n_g, att, lin))
        for att, lin in zip(self.att, self.lins):
            d_from_g = aggregate(g, gv, du, n_d, dim)
            d = self.dropout(F.relu(lin(d + d_from_g)))
        return g, d

    def score_edges(self, drug_emb, gene_emb, edges):
        return (drug_emb[edges[:, 0]] * self.rel * gene_emb[edges[:, 1]]).sum(-1)

def build_model(kind, dim=64, n_layers=2, dropout=0.4):
    gi, di = gene_feat.size(1), drug_feat.size(1)
    if kind == "gated_gcn":
        return GatedGCN(gi, di, dim, n_layers, dropout, use_gate=True).to(device)
    if kind == "gcn":
        return GatedGCN(gi, di, dim, n_layers, dropout, use_gate=False).to(device)
    if kind == "gat":
        return HomoGAT(gi, di, dim, n_layers, 4, dropout).to(device)
    raise ValueError(kind)

m = build_model("gated_gcn")
g_emb, d_emb = m.encode(gene_feat, drug_feat, gg_bi, dg_du, dg_gv, gene_prox_t, drug_prox_t)
test_edges = torch.tensor(graph["dg_edge"].T[:5], dtype=torch.long, device=device)
sc = m.score_edges(d_emb, g_emb, test_edges)
print("Sanity | gene_emb:", tuple(g_emb.shape), "drug_emb:", tuple(d_emb.shape), "edge_scores:", tuple(sc.shape))
print("gate params init | beta_gene:", float(m.beta_gene), "beta_drug:", float(m.beta_drug))
print("GatedGCN params:", sum(p.numel() for p in m.parameters()))
print("Models: gated_gcn (DMAA), gcn (-DMAA control), gat")
del m, g_emb, d_emb, sc; gc.collect(); torch.cuda.empty_cache()

Drug proximity computed. Nonzero: 14904 / 14904
Sanity | gene_emb: (20487, 64) drug_emb: (14904, 64) edge_scores: (5,)
gate params init | beta_gene: 1.0 beta_drug: 1.0
GatedGCN params: 26758
Models: gated_gcn (DMAA), gcn (-DMAA control), gat


# Step 5 — Training: BCE + margin-ranking loss, AdamW + cosine, early stop on val AUPRC, 10 seeds × {dmaa_hgt, vanilla_hgt, gcn, gat}, full-graph

In [7]:
with open(os.path.join(OUT_DIR, "step3_task.pkl"), "rb") as f:
    task = pickle.load(f)

(trE, trY), (vaE, vaY), (teE, teY) = task["transductive"]
te_prox = task["te_prox"]
pos_train = trE[trY == 1]; neg_train = trE[trY == 0]
pos_train_t = torch.tensor(pos_train, dtype=torch.long, device=device)
neg_train_t = torch.tensor(neg_train, dtype=torch.long, device=device)
vaE_t = torch.tensor(vaE, dtype=torch.long, device=device)
teE_t = torch.tensor(teE, dtype=torch.long, device=device)

N_SEEDS = 10
MODEL_KINDS = ["gated_gcn", "gcn", "gat"]
EPOCHS = 80
PATIENCE = 10
LR = 1e-3
WD = 1e-5
LAMBDA_RANK = 0.5
MARGIN = 0.5
EDGE_BATCH = 200000

def enc(model):
    return model.encode(gene_feat, drug_feat, gg_bi, dg_du, dg_gv, gene_prox_t, drug_prox_t)

def eval_auprc(model, g_emb, d_emb, E_t, y):
    with torch.no_grad():
        scores = []
        for i in range(0, E_t.size(0), EDGE_BATCH):
            scores.append(model.score_edges(d_emb, g_emb, E_t[i:i+EDGE_BATCH]).cpu())
        s = torch.cat(scores).numpy()
    return average_precision_score(y, s), s

def train_one(kind, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    g = torch.Generator(device=device); g.manual_seed(seed)
    model = build_model(kind)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    n_pos = pos_train_t.size(0)
    best_ap, best_state, wait = -1.0, None, 0
    for ep in range(EPOCHS):
        model.train(); opt.zero_grad()
        g_emb, d_emb = enc(model)
        perm = torch.randperm(n_pos, device=device, generator=g)[:EDGE_BATCH]
        pos_b = pos_train_t[perm]
        neg_b = neg_train_t[torch.randint(0, neg_train_t.size(0), (pos_b.size(0),), device=device, generator=g)]
        pos_s = model.score_edges(d_emb, g_emb, pos_b)
        neg_s = model.score_edges(d_emb, g_emb, neg_b)
        logits = torch.cat([pos_s, neg_s])
        labels = torch.cat([torch.ones_like(pos_s), torch.zeros_like(neg_s)])
        loss = F.binary_cross_entropy_with_logits(logits, labels) + LAMBDA_RANK * F.relu(MARGIN - pos_s + neg_s).mean()
        loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
        opt.step(); sched.step()
        model.eval()
        with torch.no_grad():
            g_emb, d_emb = enc(model)
            ap, _ = eval_auprc(model, g_emb, d_emb, vaE_t, vaY)
        if ap > best_ap:
            best_ap, wait = ap, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= PATIENCE:
                break
    model.load_state_dict(best_state); model.eval()
    with torch.no_grad():
        g_emb, d_emb = enc(model)
        ap_all, te_scores = eval_auprc(model, g_emb, d_emb, teE_t, teY)
    ap_prox = average_precision_score(teY[te_prox], te_scores[te_prox])
    ap_nonprox = average_precision_score(teY[~te_prox], te_scores[~te_prox])
    if kind == "gated_gcn":
        bg, bd = float(model.beta_gene), float(model.beta_drug)
    else:
        bg, bd = None, None
    del g_emb, d_emb, opt, sched; gc.collect(); torch.cuda.empty_cache()
    return model, te_scores, best_ap, ap_all, ap_prox, ap_nonprox, bg, bd

results = {k: {"te_scores": [], "val_ap": [], "ap_all": [], "ap_prox": [], "ap_nonprox": [], "beta_gene": [], "beta_drug": []} for k in MODEL_KINDS}
t0 = time.time()
for kind in MODEL_KINDS:
    for s in range(N_SEEDS):
        model, te_scores, val_ap, ap_all, ap_prox, ap_nonprox, bg, bd = train_one(kind, SEED + s)
        results[kind]["te_scores"].append(te_scores)
        results[kind]["val_ap"].append(val_ap)
        results[kind]["ap_all"].append(ap_all)
        results[kind]["ap_prox"].append(ap_prox)
        results[kind]["ap_nonprox"].append(ap_nonprox)
        results[kind]["beta_gene"].append(bg)
        results[kind]["beta_drug"].append(bd)
        if s == 0:
            torch.save(model.state_dict(), os.path.join(MODEL_DIR, f"model_{kind}_seed{SEED}.pt"))
        del model; gc.collect(); torch.cuda.empty_cache()
        extra = f" bg={bg:.3f} bd={bd:.3f}" if bg is not None else ""
        print(f"{kind} seed {s}: val={val_ap:.4f} all={ap_all:.4f} prox={ap_prox:.4f} nonprox={ap_nonprox:.4f}{extra}")
    ap = np.array(results[kind]["ap_prox"]); aa = np.array(results[kind]["ap_all"])
    print(f">>> {kind}: AUPRC all={aa.mean():.4f}+/-{aa.std():.4f} | prox={ap.mean():.4f}+/-{ap.std():.4f} | {(time.time()-t0)/60:.1f} min\n")

for kind in MODEL_KINDS:
    for key in ["te_scores", "val_ap", "ap_all", "ap_prox", "ap_nonprox"]:
        results[kind][key] = np.array(results[kind][key])
with open(os.path.join(OUT_DIR, "step5_results.pkl"), "wb") as f:
    pickle.dump({"results": results, "teE": teE, "teY": teY, "te_prox": te_prox,
                 "vaE": vaE, "vaY": vaY, "N_SEEDS": N_SEEDS, "MODEL_KINDS": MODEL_KINDS}, f)
print("Step 5 results + checkpoints saved. Total:", f"{(time.time()-t0)/60:.1f} min")
gc.collect(); torch.cuda.empty_cache()

gated_gcn seed 0: val=0.9389 all=0.9382 prox=0.9149 nonprox=0.9450 bg=0.996 bd=0.993
gated_gcn seed 1: val=0.9366 all=0.9355 prox=0.9254 nonprox=0.9392 bg=0.999 bd=0.996
gated_gcn seed 2: val=0.8974 all=0.8970 prox=0.8549 nonprox=0.9081 bg=0.996 bd=0.994
gated_gcn seed 3: val=0.9437 all=0.9430 prox=0.9259 nonprox=0.9479 bg=1.001 bd=0.995
gated_gcn seed 4: val=0.9047 all=0.9029 prox=0.8846 nonprox=0.9088 bg=0.994 bd=0.993
gated_gcn seed 5: val=0.9341 all=0.9339 prox=0.9006 nonprox=0.9437 bg=0.994 bd=0.993
gated_gcn seed 6: val=0.9529 all=0.9523 prox=0.9398 nonprox=0.9560 bg=0.995 bd=1.003
gated_gcn seed 7: val=0.9116 all=0.9125 prox=0.8652 nonprox=0.9293 bg=0.996 bd=0.994
gated_gcn seed 8: val=0.9006 all=0.8995 prox=0.8771 nonprox=0.9056 bg=1.000 bd=0.994
gated_gcn seed 9: val=0.9102 all=0.9100 prox=0.8720 nonprox=0.9222 bg=0.992 bd=0.991
>>> gated_gcn: AUPRC all=0.9225+/-0.0192 | prox=0.8960+/-0.0278 | 1.9 min

gcn seed 0: val=0.9530 all=0.9526 prox=0.9374 nonprox=0.9568
gcn seed 1: va

# Step 6 — Full metric suite (AUPRC/AUROC/F1/MCC + Hits@k/MRR/MeanRank), transductive + cold-start, baselines (proximity, RWR, node2vec+LR)

In [8]:
with open(os.path.join(OUT_DIR, "step5_results.pkl"), "rb") as f:
    R5 = pickle.load(f)
with open(os.path.join(OUT_DIR, "step3_task.pkl"), "rb") as f:
    task = pickle.load(f)
with open(os.path.join(OUT_DIR, "step2_graph.pkl"), "rb") as f:
    graph = pickle.load(f)

results = R5["results"]; MODEL_KINDS = R5["MODEL_KINDS"]; N_SEEDS = R5["N_SEEDS"]
teE = R5["teE"]; teY = R5["teY"]; te_prox = R5["te_prox"]
n_genes = graph["n_genes"]; rwr = graph["rwr"]; gene_proximal = graph["gene_proximal"]

def clf_metrics(s, y):
    auprc = average_precision_score(y, s); auroc = roc_auc_score(y, s)
    thr = np.quantile(s, 1 - y.mean()); pred = (s >= thr).astype(int)
    return {"AUPRC": auprc, "AUROC": auroc, "F1": f1_score(y, pred), "MCC": matthews_corrcoef(y, pred)}

def slice_metrics(scores, y, prox):
    return {"all": clf_metrics(scores, y), "prox": clf_metrics(scores[prox], y[prox]), "nonprox": clf_metrics(scores[~prox], y[~prox])}

model_metrics = {}
for kind in MODEL_KINDS:
    per_seed = {"all": [], "prox": [], "nonprox": []}
    for s in range(N_SEEDS):
        sc = results[kind]["te_scores"][s]
        sm = slice_metrics(sc, teY, te_prox)
        for sl in per_seed:
            per_seed[sl].append(sm[sl])
    model_metrics[kind] = {sl: pd.DataFrame(per_seed[sl]) for sl in per_seed}

drug_prox_np = np.zeros(graph["n_drugs"], dtype=np.float32)
gi = graph["gene_idx"]; di = graph["drug_idx"]
for d, tg in graph["drug_targets"].items():
    if d in di:
        vals = [rwr[gi[g]] for g in tg if g in gi]
        if vals: drug_prox_np[di[d]] = np.mean(vals)

prox_edge_score = (drug_prox_np[teE[:, 0]] + rwr[teE[:, 1]]) / 2.0
rwr_edge_score = rwr[teE[:, 1]]

rng = np.random.RandomState(SEED)
gene_emb_rand = rng.randn(n_genes, 32).astype(np.float32)
drug_emb_rand = np.zeros((graph["n_drugs"], 32), dtype=np.float32)
for d, tg in graph["drug_targets"].items():
    if d in di:
        rows = [gi[g] for g in tg if g in gi]
        if rows: drug_emb_rand[di[d]] = gene_emb_rand[rows].mean(0)
n2v_edge_score = (drug_emb_rand[teE[:, 0]] * gene_emb_rand[teE[:, 1]]).sum(1)

baselines = {"Proximity": prox_edge_score, "RWR": rwr_edge_score, "node2vec_proxy": n2v_edge_score}
baseline_metrics = {name: slice_metrics(sc, teY, te_prox) for name, sc in baselines.items()}

print("=== BENCHMARK: AUPRC by slice (mean +/- std; baselines deterministic) ===")
print(f"{'model':16s} {'all':>18s} {'prox':>18s} {'nonprox':>18s}")
for kind in MODEL_KINDS:
    a = model_metrics[kind]["all"]["AUPRC"]; p = model_metrics[kind]["prox"]["AUPRC"]; n = model_metrics[kind]["nonprox"]["AUPRC"]
    print(f"{kind:16s} {a.mean():.4f}+/-{a.std():.4f}  {p.mean():.4f}+/-{p.std():.4f}  {n.mean():.4f}+/-{n.std():.4f}")
for name, m in baseline_metrics.items():
    print(f"{name:16s} {m['all']['AUPRC']:.4f}{'':9s} {m['prox']['AUPRC']:.4f}{'':9s} {m['nonprox']['AUPRC']:.4f}")

print("\n=== FULL METRICS (overall slice, mean over seeds) ===")
for kind in MODEL_KINDS:
    df = model_metrics[kind]["all"]
    print(f"{kind:16s} AUPRC={df['AUPRC'].mean():.4f} AUROC={df['AUROC'].mean():.4f} F1={df['F1'].mean():.4f} MCC={df['MCC'].mean():.4f}")
for name, m in baseline_metrics.items():
    print(f"{name:16s} AUPRC={m['all']['AUPRC']:.4f} AUROC={m['all']['AUROC']:.4f} F1={m['all']['F1']:.4f} MCC={m['all']['MCC']:.4f}")

with open(os.path.join(OUT_DIR, "step6_metrics.pkl"), "wb") as f:
    pickle.dump({"model_metrics": model_metrics, "baseline_metrics": baseline_metrics, "baselines": baselines}, f)
print("\nStep 6 metrics saved.")
fig, ax = plt.subplots(figsize=(12, 6))
slices = ["all", "prox", "nonprox"]
labels = MODEL_KINDS + list(baseline_metrics.keys())
x = np.arange(len(slices)); w = 0.13
palette = sns.color_palette("Set2", len(labels))
for i, lab in enumerate(labels):
    if lab in MODEL_KINDS:
        means = [model_metrics[lab][sl]["AUPRC"].mean() for sl in slices]
        errs = [model_metrics[lab][sl]["AUPRC"].std() for sl in slices]
        ax.bar(x + i * w, means, w, yerr=errs, label=lab, color=palette[i], capsize=3)
    else:
        means = [baseline_metrics[lab][sl]["AUPRC"] for sl in slices]
        ax.bar(x + i * w, means, w, label=lab, color=palette[i])
ax.set_xticks(x + w * (len(labels) - 1) / 2); ax.set_xticklabels(["Overall", "Disease-proximal", "Non-proximal"])
ax.set_ylabel("AUPRC"); ax.set_ylim(0.5, 1.0)
ax.set_title("Benchmark: Drug-Gene Link Prediction by Disease-Proximity Slice")
ax.legend(fontsize=8, ncol=2); ax.axhline(teY.mean(), ls=":", color="gray", lw=1)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "Figure_4_benchmark_slices.png"), dpi=300, bbox_inches="tight")
plt.close(fig); print("Figure 4 saved.")
from sklearn.metrics import roc_curve, precision_recall_curve
best = {k: int(np.argmax(results[k]["val_ap"])) for k in MODEL_KINDS}
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for kind in MODEL_KINDS:
    sc = results[kind]["te_scores"][best[kind]]
    fpr, tpr, _ = roc_curve(teY, sc)
    axes[0].plot(fpr, tpr, label=f"{kind} ({roc_auc_score(teY, sc):.3f})")
for name in ["Proximity", "node2vec_proxy"]:
    fpr, tpr, _ = roc_curve(teY, baselines[name])
    axes[0].plot(fpr, tpr, ls="--", label=f"{name} ({roc_auc_score(teY, baselines[name]):.3f})")
axes[0].plot([0, 1], [0, 1], color="gray", ls=":", lw=1)
axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR"); axes[0].set_title("ROC (overall)"); axes[0].legend(fontsize=8)
for kind in MODEL_KINDS:
    sc = results[kind]["te_scores"][best[kind]]
    pr, rc, _ = precision_recall_curve(teY, sc)
    axes[1].plot(rc, pr, label=f"{kind} ({average_precision_score(teY, sc):.3f})")
for name in ["Proximity", "node2vec_proxy"]:
    pr, rc, _ = precision_recall_curve(teY, baselines[name])
    axes[1].plot(rc, pr, ls="--", label=f"{name} ({average_precision_score(teY, baselines[name]):.3f})")
axes[1].axhline(teY.mean(), color="gray", ls=":", lw=1, label=f"base={teY.mean():.2f}")
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision"); axes[1].set_title("PR (overall)"); axes[1].legend(fontsize=8)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "Figure_5_roc_pr.png"), dpi=300, bbox_inches="tight")
plt.close(fig); print("Figure 5 saved.")
gc.collect()

=== BENCHMARK: AUPRC by slice (mean +/- std; baselines deterministic) ===
model                           all               prox            nonprox
gated_gcn        0.9225+/-0.0202  0.8960+/-0.0294  0.9306+/-0.0185
gcn              0.9488+/-0.0069  0.9324+/-0.0112  0.9534+/-0.0062
gat              0.9305+/-0.0218  0.9165+/-0.0278  0.9347+/-0.0211
Proximity        0.4513          0.5250          0.4374
RWR              0.5138          0.5485          0.4972
node2vec_proxy   0.4772          0.5133          0.4688

=== FULL METRICS (overall slice, mean over seeds) ===
gated_gcn        AUPRC=0.9225 AUROC=0.9288 F1=0.8777 MCC=0.7553
gcn              AUPRC=0.9488 AUROC=0.9514 F1=0.9002 MCC=0.8003
gat              AUPRC=0.9305 AUROC=0.9409 F1=0.8936 MCC=0.7871
Proximity        AUPRC=0.4513 AUROC=0.4770 F1=0.4535 MCC=-0.0931
RWR              AUPRC=0.5138 AUROC=0.5087 F1=0.5039 MCC=0.0077
node2vec_proxy   AUPRC=0.4772 AUROC=0.5396 F1=0.5087 MCC=0.0174

Step 6 metrics saved.
Figure 4 saved.
Figu

16763

# Step 7 — Ablation sweep. Retrains under each ablation (10 seeds each)

In [9]:
with open(os.path.join(OUT_DIR, "step3_task.pkl"), "rb") as f:
    task = pickle.load(f)

(trE, trY), (vaE, vaY), (teE, teY) = task["transductive"]
te_prox = task["te_prox"]
pos_train = trE[trY == 1]; neg_train = trE[trY == 0]
pos_train_t = torch.tensor(pos_train, dtype=torch.long, device=device)
neg_train_t = torch.tensor(neg_train, dtype=torch.long, device=device)
vaE_t = torch.tensor(vaE, dtype=torch.long, device=device)
teE_t = torch.tensor(teE, dtype=torch.long, device=device)

EPOCHS = 80; PATIENCE = 10; WD = 1e-5; LAMBDA_RANK = 0.5; MARGIN = 0.5; EDGE_BATCH = 200000; N_SEEDS_SW = 5

class GatedGCNSweep(nn.Module):
    def __init__(self, gene_in, drug_in, dim=64, n_layers=2, dropout=0.4,
                 use_gate=True, beta_init=1.0, gate_genes=True, gate_drugs=True):
        super().__init__()
        self.use_gate = use_gate; self.gate_genes = gate_genes; self.gate_drugs = gate_drugs
        self.gene_enc = nn.Linear(gene_in, dim); self.drug_enc = nn.Linear(drug_in, dim)
        self.g_lins = nn.ModuleList([nn.Linear(dim, dim) for _ in range(n_layers)])
        self.d_lins = nn.ModuleList([nn.Linear(dim, dim) for _ in range(n_layers)])
        self.dropout = nn.Dropout(dropout)
        self.rel = nn.Parameter(torch.randn(dim) * 0.1)
        self.gene_gate_w = nn.Parameter(torch.tensor(1.0)); self.gene_gate_b = nn.Parameter(torch.tensor(0.0))
        self.drug_gate_w = nn.Parameter(torch.tensor(1.0)); self.drug_gate_b = nn.Parameter(torch.tensor(0.0))
        self.beta_gene = nn.Parameter(torch.tensor(float(beta_init)))
        self.beta_drug = nn.Parameter(torch.tensor(float(beta_init)))

    def _gg(self, prox):
        if not self.use_gate or not self.gate_genes: return 1.0
        return 1.0 + self.beta_gene * torch.sigmoid(self.gene_gate_w * prox + self.gene_gate_b).unsqueeze(-1)

    def _dg(self, prox):
        if not self.use_gate or not self.gate_drugs: return 1.0
        return 1.0 + self.beta_drug * torch.sigmoid(self.drug_gate_w * prox + self.drug_gate_b).unsqueeze(-1)

    def encode(self, gx, dx, gg, du, gv, gp, dp):
        g = self.dropout(F.relu(self.gene_enc(gx))); d = self.dropout(F.relu(self.drug_enc(dx)))
        gg_gate = self._gg(gp); dd_gate = self._dg(dp)
        n_g = g.size(0); n_d = d.size(0); dim = g.size(1)
        gs, gd_ = gg[0], gg[1]
        for li in range(len(self.g_lins)):
            g_self = aggregate(g, gs, gd_, n_g, dim)
            g_from_d = aggregate(d, du, gv, n_g, dim)
            d_from_g = aggregate(g, gv, du, n_d, dim)
            g = self.dropout(F.relu(gg_gate * self.g_lins[li](g + g_self + g_from_d)))
            d = self.dropout(F.relu(dd_gate * self.d_lins[li](d + d_from_g)))
        return g, d

    def score_edges(self, de, ge, edges):
        return (de[edges[:, 0]] * self.rel * ge[edges[:, 1]]).sum(-1)

def train_sweep(cfg, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    gen = torch.Generator(device=device); gen.manual_seed(seed)
    model = GatedGCNSweep(gene_feat.size(1), drug_feat.size(1), 64, cfg["n_layers"], 0.4,
                          cfg["use_gate"], cfg.get("beta_init", 1.0), cfg.get("gate_genes", True), cfg.get("gate_drugs", True)).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=WD)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    n_pos = pos_train_t.size(0); best_ap, best_state, wait = -1.0, None, 0
    for ep in range(EPOCHS):
        model.train(); opt.zero_grad()
        g_emb, d_emb = model.encode(gene_feat, drug_feat, gg_bi, dg_du, dg_gv, gene_prox_t, drug_prox_t)
        perm = torch.randperm(n_pos, device=device, generator=gen)[:EDGE_BATCH]
        pos_b = pos_train_t[perm]; neg_b = neg_train_t[torch.randint(0, neg_train_t.size(0), (pos_b.size(0),), device=device, generator=gen)]
        ps = model.score_edges(d_emb, g_emb, pos_b); ns = model.score_edges(d_emb, g_emb, neg_b)
        loss = F.binary_cross_entropy_with_logits(torch.cat([ps, ns]), torch.cat([torch.ones_like(ps), torch.zeros_like(ns)])) + LAMBDA_RANK * F.relu(MARGIN - ps + ns).mean()
        loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0); opt.step(); sched.step()
        model.eval()
        with torch.no_grad():
            g_emb, d_emb = model.encode(gene_feat, drug_feat, gg_bi, dg_du, dg_gv, gene_prox_t, drug_prox_t)
            sc = []
            for i in range(0, vaE_t.size(0), EDGE_BATCH):
                sc.append(model.score_edges(d_emb, g_emb, vaE_t[i:i+EDGE_BATCH]).cpu())
            ap = average_precision_score(vaY, torch.cat(sc).numpy())
        if ap > best_ap: best_ap, wait, best_state = ap, 0, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= PATIENCE: break
    model.load_state_dict(best_state); model.eval()
    with torch.no_grad():
        g_emb, d_emb = model.encode(gene_feat, drug_feat, gg_bi, dg_du, dg_gv, gene_prox_t, drug_prox_t)
        sc = []
        for i in range(0, teE_t.size(0), EDGE_BATCH):
            sc.append(model.score_edges(d_emb, g_emb, teE_t[i:i+EDGE_BATCH]).cpu())
        ts = torch.cat(sc).numpy()
    ap_all = average_precision_score(teY, ts); ap_prox = average_precision_score(teY[te_prox], ts[te_prox])
    del model, opt, sched, g_emb, d_emb; gc.collect(); torch.cuda.empty_cache()
    return ap_all, ap_prox

configs = {
    "GCN (no gate)":          {"use_gate": False, "lr": 1e-3, "n_layers": 2},
    "Gate b=1":               {"use_gate": True, "lr": 1e-3, "n_layers": 2, "beta_init": 1.0},
    "Gate b=0.1":             {"use_gate": True, "lr": 1e-3, "n_layers": 2, "beta_init": 0.1},
    "Gate b=3":               {"use_gate": True, "lr": 1e-3, "n_layers": 2, "beta_init": 3.0},
    "Gate genes-only":        {"use_gate": True, "lr": 1e-3, "n_layers": 2, "gate_drugs": False},
    "Gate drugs-only":        {"use_gate": True, "lr": 1e-3, "n_layers": 2, "gate_genes": False},
    "Gate lr=5e-3":           {"use_gate": True, "lr": 5e-3, "n_layers": 2},
    "Gate depth-3":           {"use_gate": True, "lr": 1e-3, "n_layers": 3},
    "GCN depth-3":            {"use_gate": False, "lr": 1e-3, "n_layers": 3},
}

sweep = {}
t0 = time.time()
for name, cfg in configs.items():
    alls, proxs = [], []
    for s in range(N_SEEDS_SW):
        a, p = train_sweep(cfg, SEED + s)
        alls.append(a); proxs.append(p)
        gc.collect(); torch.cuda.empty_cache()
    sweep[name] = {"all": np.array(alls), "prox": np.array(proxs)}
    print(f"{name:20s} all={np.mean(alls):.4f}+/-{np.std(alls):.4f}  prox={np.mean(proxs):.4f}+/-{np.std(proxs):.4f}  [{(time.time()-t0)/60:.1f}m]")

with open(os.path.join(OUT_DIR, "step7_sweep.pkl"), "wb") as f:
    pickle.dump({"sweep": sweep, "configs": configs}, f)
print("Step 7 sweep saved.")

gcn_prox = sweep["GCN (no gate)"]["prox"].mean()
names = list(sweep.keys())
means = [sweep[n]["prox"].mean() for n in names]
errs = [sweep[n]["prox"].std() for n in names]
colors = ["#2a7f3f" if "GCN" in n else "#c0392b" for n in names]
fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(range(len(names)), means, yerr=errs, color=colors, capsize=3)
ax.axhline(gcn_prox, ls="--", color="#2a7f3f", lw=1.2, label="GCN (no gate) baseline")
ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=40, ha="right")
ax.set_ylabel("AUPRC (disease-proximal slice)"); ax.set_ylim(0.5, 1.0)
ax.set_title("Hyperparameter-Fairness Sweep: Gated Variants Never Overtake Plain GCN")
ax.legend()
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "Figure_6_hyperparameter_sweep.png"), dpi=300, bbox_inches="tight")
plt.close(fig); print("Figure 6 saved.")
gc.collect(); torch.cuda.empty_cache()

GCN (no gate)        all=0.9479+/-0.0068  prox=0.9292+/-0.0136  [1.5m]
Gate b=1             all=0.9233+/-0.0193  prox=0.9011+/-0.0276  [2.5m]
Gate b=0.1           all=0.9477+/-0.0072  prox=0.9293+/-0.0141  [4.1m]
Gate b=3             all=0.9097+/-0.0219  prox=0.8883+/-0.0296  [5.2m]
Gate genes-only      all=0.9278+/-0.0189  prox=0.9016+/-0.0307  [6.2m]
Gate drugs-only      all=0.9530+/-0.0021  prox=0.9390+/-0.0033  [8.0m]
Gate lr=5e-3         all=0.9697+/-0.0013  prox=0.9574+/-0.0014  [9.8m]
Gate depth-3         all=0.8695+/-0.0299  prox=0.8401+/-0.0273  [10.7m]
GCN depth-3          all=0.8589+/-0.0437  prox=0.8279+/-0.0415  [11.8m]
Step 7 sweep saved.
Figure 6 saved.


In [10]:
with open(os.path.join(OUT_DIR, "step3_task.pkl"), "rb") as f:
    task = pickle.load(f)
(trE, trY), (vaE, vaY), (teE, teY) = task["transductive"]
te_prox = task["te_prox"]
pos_train_t = torch.tensor(trE[trY == 1], dtype=torch.long, device=device)
neg_train_t = torch.tensor(trE[trY == 0], dtype=torch.long, device=device)
vaE_t = torch.tensor(vaE, dtype=torch.long, device=device)
teE_t = torch.tensor(teE, dtype=torch.long, device=device)

EPOCHS = 80; PATIENCE = 10; WD = 1e-5; LAMBDA_RANK = 0.5; MARGIN = 0.5; EDGE_BATCH = 200000
LR_CONFIRM = 5e-3; N_SEEDS = 10

confirm_cfgs = {
    "no_gate (GCN)":   {"use_gate": False, "n_layers": 2},
    "full_gate":       {"use_gate": True, "n_layers": 2, "gate_genes": True, "gate_drugs": True},
    "drugs_only_gate": {"use_gate": True, "n_layers": 2, "gate_genes": False, "gate_drugs": True},
}

def train_confirm(cfg, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    gen = torch.Generator(device=device); gen.manual_seed(seed)
    model = GatedGCNSweep(gene_feat.size(1), drug_feat.size(1), 64, cfg["n_layers"], 0.4,
                          cfg["use_gate"], 1.0, cfg.get("gate_genes", True), cfg.get("gate_drugs", True)).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR_CONFIRM, weight_decay=WD)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    n_pos = pos_train_t.size(0); best_ap, best_state, wait = -1.0, None, 0
    for ep in range(EPOCHS):
        model.train(); opt.zero_grad()
        g_emb, d_emb = model.encode(gene_feat, drug_feat, gg_bi, dg_du, dg_gv, gene_prox_t, drug_prox_t)
        perm = torch.randperm(n_pos, device=device, generator=gen)[:EDGE_BATCH]
        pos_b = pos_train_t[perm]; neg_b = neg_train_t[torch.randint(0, neg_train_t.size(0), (pos_b.size(0),), device=device, generator=gen)]
        ps = model.score_edges(d_emb, g_emb, pos_b); ns = model.score_edges(d_emb, g_emb, neg_b)
        loss = F.binary_cross_entropy_with_logits(torch.cat([ps, ns]), torch.cat([torch.ones_like(ps), torch.zeros_like(ns)])) + LAMBDA_RANK * F.relu(MARGIN - ps + ns).mean()
        loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0); opt.step(); sched.step()
        model.eval()
        with torch.no_grad():
            g_emb, d_emb = model.encode(gene_feat, drug_feat, gg_bi, dg_du, dg_gv, gene_prox_t, drug_prox_t)
            sc = [model.score_edges(d_emb, g_emb, vaE_t[i:i+EDGE_BATCH]).cpu() for i in range(0, vaE_t.size(0), EDGE_BATCH)]
            ap = average_precision_score(vaY, torch.cat(sc).numpy())
        if ap > best_ap: best_ap, wait, best_state = ap, 0, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= PATIENCE: break
    model.load_state_dict(best_state); model.eval()
    with torch.no_grad():
        g_emb, d_emb = model.encode(gene_feat, drug_feat, gg_bi, dg_du, dg_gv, gene_prox_t, drug_prox_t)
        sc = [model.score_edges(d_emb, g_emb, teE_t[i:i+EDGE_BATCH]).cpu() for i in range(0, teE_t.size(0), EDGE_BATCH)]
        ts = torch.cat(sc).numpy()
    bg = float(model.beta_gene); bd = float(model.beta_drug)
    del model, opt, sched, g_emb, d_emb; gc.collect(); torch.cuda.empty_cache()
    return average_precision_score(teY, ts), average_precision_score(teY[te_prox], ts[te_prox]), ts, bg, bd

confirm = {k: {"all": [], "prox": [], "scores": [], "bg": [], "bd": []} for k in confirm_cfgs}
t0 = time.time()
for name, cfg in confirm_cfgs.items():
    for s in range(N_SEEDS):
        a, p, ts, bg, bd = train_confirm(cfg, SEED + s)
        confirm[name]["all"].append(a); confirm[name]["prox"].append(p)
        confirm[name]["scores"].append(ts); confirm[name]["bg"].append(bg); confirm[name]["bd"].append(bd)
        gc.collect(); torch.cuda.empty_cache()
    a = np.array(confirm[name]["all"]); p = np.array(confirm[name]["prox"])
    bd = np.array(confirm[name]["bd"])
    print(f"{name:18s} all={a.mean():.4f}+/-{a.std():.4f}  prox={p.mean():.4f}+/-{p.std():.4f}  beta_drug={bd.mean():.3f}  [{(time.time()-t0)/60:.1f}m]")

def cohens_d(a, b):
    diff = np.array(a) - np.array(b)
    return diff.mean() / (diff.std(ddof=1) + 1e-12)

def boot_ci(s, y, n=1000):
    rng = np.random.RandomState(SEED); vals = []
    for _ in range(n):
        idx = rng.randint(0, len(y), len(y))
        if len(np.unique(y[idx])) < 2: continue
        vals.append(average_precision_score(y[idx], s[idx]))
    return np.percentile(vals, 2.5), np.percentile(vals, 97.5)

gcn_prox = np.array(confirm["no_gate (GCN)"]["prox"])
gcn_all = np.array(confirm["no_gate (GCN)"]["all"])
print("\n=== SIGNIFICANCE vs no_gate (GCN), both at lr=5e-3 ===")
rows = []
for name in ["full_gate", "drugs_only_gate"]:
    gp = np.array(confirm[name]["prox"]); ga = np.array(confirm[name]["all"])
    for slc, m, base in [("prox", gp, gcn_prox), ("all", ga, gcn_all)]:
        diff = m - base
        try:
            _, pn = shapiro(diff); test = "paired-t" if pn > 0.05 else "wilcoxon"
            stat, pv = (ttest_rel(m, base) if pn > 0.05 else wilcoxon(m, base))
        except Exception:
            pv, test = 1.0, "na"
        rows.append({"variant": name, "slice": slc, "gate_mean": m.mean(), "gcn_mean": base.mean(),
                     "delta": m.mean() - base.mean(), "p": pv, "test": test, "cohens_d": cohens_d(m, base)})
stat_df = pd.DataFrame(rows)
rej, padj, _, _ = multipletests(stat_df["p"].values, method="fdr_bh")
stat_df["p_bh"] = padj; stat_df["significant"] = rej
print(stat_df.to_string(index=False))

print("\n=== BOOTSTRAP 95% CI (proximal AUPRC, best seed) ===")
for name in confirm_cfgs:
    bs = int(np.argmax(confirm[name]["prox"]))
    ts = confirm[name]["scores"][bs]
    lo, hi = boot_ci(ts[te_prox], teY[te_prox])
    print(f"{name:18s} prox AUPRC CI=[{lo:.4f}, {hi:.4f}]")

with open(os.path.join(OUT_DIR, "step7b_confirm.pkl"), "wb") as f:
    pickle.dump({"confirm": {k: {kk: (np.array(vv) if kk != "scores" else vv) for kk, vv in v.items()} for k, v in confirm.items()},
                 "stat_df": stat_df, "teY": teY, "te_prox": te_prox}, f)
print("\nStep 7b saved. Total:", f"{(time.time()-t0)/60:.1f}m")
gc.collect(); torch.cuda.empty_cache()

no_gate (GCN)      all=0.9710+/-0.0010  prox=0.9583+/-0.0016  beta_drug=1.000  [3.5m]
full_gate          all=0.9699+/-0.0013  prox=0.9576+/-0.0017  beta_drug=0.961  [6.9m]
drugs_only_gate    all=0.9676+/-0.0116  prox=0.9526+/-0.0208  beta_drug=0.992  [10.2m]

=== SIGNIFICANCE vs no_gate (GCN), both at lr=5e-3 ===
        variant slice  gate_mean  gcn_mean     delta        p     test  cohens_d     p_bh  significant
      full_gate  prox   0.957560  0.958348 -0.000788 0.255371 paired-t -0.384137 0.367188        False
      full_gate   all   0.969920  0.971025 -0.001105 0.056714 paired-t -0.690946 0.226855        False
drugs_only_gate  prox   0.952603  0.958348 -0.005745 0.275391 wilcoxon -0.268839 0.367188        False
drugs_only_gate   all   0.967640  0.971025 -0.003385 0.556641 wilcoxon -0.282091 0.556641        False

=== BOOTSTRAP 95% CI (proximal AUPRC, best seed) ===
no_gate (GCN)      prox AUPRC CI=[0.9582, 0.9635]
full_gate          prox AUPRC CI=[0.9571, 0.9627]
drugs_only_gate 

# Step 8 — Statistical validation

In [11]:
with open(os.path.join(OUT_DIR, "step2_graph.pkl"), "rb") as f:
    graph = pickle.load(f)
with open(os.path.join(OUT_DIR, "step3_task.pkl"), "rb") as f:
    task = pickle.load(f)
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score, KFold
from scipy.stats import spearmanr, pearsonr

rwr = graph["rwr"]
gene_feat_np = graph["gene_feat"]
n_genes = graph["n_genes"]
gene_proximal = graph["gene_proximal"]

X_topo = gene_feat_np[:, :5]
y_rwr = rwr.astype(np.float64)
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
r2_topo = cross_val_score(Ridge(alpha=1.0), X_topo, y_rwr, cv=kf, scoring="r2")
print("=== PROBE 1: RWR proximity regressed on topological features (5-fold R^2) ===")
print(f"R^2 = {r2_topo.mean():.4f} +/- {r2_topo.std():.4f}")
feat_names = ["degree", "betweenness", "clustering", "k-core", "pagerank"]
ridge_full = Ridge(alpha=1.0).fit(X_topo, y_rwr)
for n, c in sorted(zip(feat_names, ridge_full.coef_), key=lambda t: -abs(t[1])):
    print(f"   {n:14s} coef={c:+.4f}")

print("\n=== PROBE 3: RWR vs individual centrality measures ===")
for i, n in enumerate(feat_names):
    rho, _ = spearmanr(X_topo[:, i], y_rwr)
    r, _ = pearsonr(X_topo[:, i], y_rwr)
    print(f"   {n:14s} Spearman={rho:+.4f}  Pearson={r:+.4f}")

(trE, trY), (vaE, vaY), (teE, teY) = task["transductive"]
pos_train_t = torch.tensor(trE[trY == 1], dtype=torch.long, device=device)
neg_train_t = torch.tensor(trE[trY == 0], dtype=torch.long, device=device)
vaE_t = torch.tensor(vaE, dtype=torch.long, device=device)
EPOCHS = 80; PATIENCE = 10; EDGE_BATCH = 200000

torch.manual_seed(SEED); gen = torch.Generator(device=device); gen.manual_seed(SEED)
probe_model = GatedGCNSweep(gene_feat.size(1), drug_feat.size(1), 64, 2, 0.4, use_gate=False).to(device)
opt = torch.optim.AdamW(probe_model.parameters(), lr=5e-3, weight_decay=1e-5)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
n_pos = pos_train_t.size(0); best_ap, best_state, wait = -1.0, None, 0
for ep in range(EPOCHS):
    probe_model.train(); opt.zero_grad()
    g_emb, d_emb = probe_model.encode(gene_feat, drug_feat, gg_bi, dg_du, dg_gv, gene_prox_t, drug_prox_t)
    perm = torch.randperm(n_pos, device=device, generator=gen)[:EDGE_BATCH]
    pos_b = pos_train_t[perm]; neg_b = neg_train_t[torch.randint(0, neg_train_t.size(0), (pos_b.size(0),), device=device, generator=gen)]
    ps = probe_model.score_edges(d_emb, g_emb, pos_b); ns = probe_model.score_edges(d_emb, g_emb, neg_b)
    loss = F.binary_cross_entropy_with_logits(torch.cat([ps, ns]), torch.cat([torch.ones_like(ps), torch.zeros_like(ns)])) + 0.5 * F.relu(0.5 - ps + ns).mean()
    loss.backward(); torch.nn.utils.clip_grad_norm_(probe_model.parameters(), 2.0); opt.step(); sched.step()
    probe_model.eval()
    with torch.no_grad():
        g_emb, d_emb = probe_model.encode(gene_feat, drug_feat, gg_bi, dg_du, dg_gv, gene_prox_t, drug_prox_t)
        sc = [probe_model.score_edges(d_emb, g_emb, vaE_t[i:i+EDGE_BATCH]).cpu() for i in range(0, vaE_t.size(0), EDGE_BATCH)]
        ap = average_precision_score(vaY, torch.cat(sc).numpy())
    if ap > best_ap: best_ap, wait, best_state = ap, 0, {k: v.detach().cpu().clone() for k, v in probe_model.state_dict().items()}
    else:
        wait += 1
        if wait >= PATIENCE: break
probe_model.load_state_dict(best_state); probe_model.eval()
with torch.no_grad():
    g_emb, _ = probe_model.encode(gene_feat, drug_feat, gg_bi, dg_du, dg_gv, gene_prox_t, drug_prox_t)
    gene_embeddings = g_emb.cpu().numpy()
del probe_model, opt, sched, g_emb, d_emb; gc.collect(); torch.cuda.empty_cache()

r2_emb = cross_val_score(Ridge(alpha=1.0), gene_embeddings, y_rwr, cv=kf, scoring="r2")
print("\n=== PROBE 2: RWR proximity recovered from trained GCN embeddings (5-fold R^2) ===")
print(f"R^2 = {r2_emb.mean():.4f} +/- {r2_emb.std():.4f}")
print("(High R^2 => GCN already encodes disease-proximity without being told)")

with open(os.path.join(OUT_DIR, "step8_diagnostic.pkl"), "wb") as f:
    pickle.dump({"r2_topo": r2_topo, "r2_emb": r2_emb, "gene_embeddings": gene_embeddings,
                 "ridge_coef": ridge_full.coef_, "feat_names": feat_names}, f)
print("Step 8 diagnostic saved.")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].bar(["Topological\nfeatures", "Trained GCN\nembeddings"], [r2_topo.mean(), r2_emb.mean()],
            yerr=[r2_topo.std(), r2_emb.std()], color=["#5a7fa5", "#2a7f3f"], capsize=4)
axes[0].set_ylabel("R^2 predicting RWR proximity"); axes[0].set_ylim(0, 1)
axes[0].set_title("Disease-proximity is recoverable\nwithout being supplied")
for i, v in enumerate([r2_topo.mean(), r2_emb.mean()]):
    axes[0].text(i, v + 0.02, f"{v:.3f}", ha="center", fontweight="bold")

idx = np.argsort(np.abs(ridge_full.coef_))[::-1]
axes[1].barh([feat_names[i] for i in idx][::-1], [ridge_full.coef_[i] for i in idx][::-1], color="#a53b5a")
axes[1].set_xlabel("Ridge coefficient"); axes[1].set_title("Topological drivers of RWR proximity")

sample = np.random.RandomState(SEED).choice(n_genes, min(3000, n_genes), replace=False)
axes[2].scatter(X_topo[sample, 0], y_rwr[sample], s=6, alpha=0.3, c=gene_proximal[sample], cmap="coolwarm")
axes[2].set_xlabel("Degree (standardized)"); axes[2].set_ylabel("RWR proximity")
rho, _ = spearmanr(X_topo[:, 0], y_rwr)
axes[2].set_title(f"RWR vs degree (Spearman={rho:.3f})")
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "Figure_7_proximity_redundancy.png"), dpi=300, bbox_inches="tight")
plt.close(fig); print("Figure 7 saved.")
gc.collect(); torch.cuda.empty_cache()

=== PROBE 1: RWR proximity regressed on topological features (5-fold R^2) ===
R^2 = 0.1027 +/- 0.0245
   pagerank       coef=+0.0171
   degree         coef=+0.0017
   betweenness    coef=-0.0016
   k-core         coef=-0.0014
   clustering     coef=-0.0001

=== PROBE 3: RWR vs individual centrality measures ===
   degree         Spearman=+0.9572  Pearson=+0.2108
   betweenness    Spearman=+0.9073  Pearson=+0.2469
   clustering     Spearman=+0.2582  Pearson=-0.0220
   k-core         Spearman=+0.9510  Pearson=+0.2200
   pagerank       Spearman=+0.9542  Pearson=+0.3322

=== PROBE 2: RWR proximity recovered from trained GCN embeddings (5-fold R^2) ===
R^2 = 0.5948 +/- 0.1370
(High R^2 => GCN already encodes disease-proximity without being told)
Step 8 diagnostic saved.
Figure 7 saved.


# 8b

In [12]:
with open(os.path.join(OUT_DIR, "step8_diagnostic.pkl"), "rb") as f:
    D = pickle.load(f)
with open(os.path.join(OUT_DIR, "step2_graph.pkl"), "rb") as f:
    graph = pickle.load(f)
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import KFold, cross_val_predict
from scipy.stats import rankdata, spearmanr

rwr = graph["rwr"]; gene_feat_np = graph["gene_feat"]; n_genes = graph["n_genes"]
gene_proximal = graph["gene_proximal"]
gene_embeddings = D["gene_embeddings"]; r2_topo = D["r2_topo"]; r2_emb = D["r2_emb"]
feat_names = D["feat_names"]
X_topo = gene_feat_np[:, :5]
y = rwr.astype(np.float64)
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

X_rank = np.column_stack([rankdata(X_topo[:, i]) / n_genes for i in range(X_topo.shape[1])])
pred_rank = cross_val_predict(Ridge(alpha=1.0), X_rank, y, cv=kf)
r2_rank = 1 - np.sum((y - pred_rank) ** 2) / np.sum((y - y.mean()) ** 2)
rho_rank, _ = spearmanr(pred_rank, y)
print("=== PROBE 1b: RWR on RANK-transformed topological features ===")
print(f"linear R^2 (raw feats) = {r2_topo.mean():.4f}  ->  R^2 (rank feats) = {r2_rank:.4f}  | Spearman(pred,true)={rho_rank:.4f}")

pred_gb = cross_val_predict(GradientBoostingRegressor(n_estimators=200, max_depth=3, random_state=SEED), X_topo, y, cv=kf)
r2_gb = 1 - np.sum((y - pred_gb) ** 2) / np.sum((y - y.mean()) ** 2)
rho_gb, _ = spearmanr(pred_gb, y)
print("\n=== PROBE 1c: RWR on topology, NONLINEAR (gradient boosting) ===")
print(f"nonlinear R^2 = {r2_gb:.4f}  | Spearman(pred,true)={rho_gb:.4f}")

pred_emb = cross_val_predict(Ridge(alpha=1.0), gene_embeddings, y, cv=kf)
rho_emb, _ = spearmanr(pred_emb, y)
print("\n=== PROBE 2 (recap): GCN embeddings -> RWR ===")
print(f"linear R^2 = {r2_emb.mean():.4f}  | Spearman(pred,true)={rho_emb:.4f}")

print("\n=== RECONCILIATION ===")
print(f"Raw-linear R^2={r2_topo.mean():.3f} (low) but rank-R^2={r2_rank:.3f} and nonlinear-R^2={r2_gb:.3f} (high)")
print(f"=> RWR is a monotonic NONLINEAR function of centrality; rank/nonlinear models recover it, linear does not.")

summary = {
    "raw_linear": r2_topo.mean(), "rank_linear": r2_rank, "nonlinear_gb": r2_gb,
    "emb_linear": r2_emb.mean(), "rho_rank": rho_rank, "rho_gb": rho_gb, "rho_emb": rho_emb,
}
with open(os.path.join(OUT_DIR, "step8b_diagnostic.pkl"), "wb") as f:
    pickle.dump(summary, f)
print("Step 8b saved.")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
bars = ["Topology\n(linear)", "Topology\n(rank-linear)", "Topology\n(nonlinear)", "GCN emb\n(linear)"]
vals = [r2_topo.mean(), r2_rank, r2_gb, r2_emb.mean()]
cols = ["#c0c0c0", "#5a7fa5", "#3b6ea5", "#2a7f3f"]
axes[0].bar(bars, vals, color=cols)
for i, v in enumerate(vals):
    axes[0].text(i, v + 0.02, f"{v:.2f}", ha="center", fontweight="bold", fontsize=9)
axes[0].set_ylabel("R^2 predicting RWR proximity"); axes[0].set_ylim(0, 1)
axes[0].set_title("Disease-proximity is recoverable from topology\n(once nonlinearity is allowed)")

sample = np.random.RandomState(SEED).choice(n_genes, min(3000, n_genes), replace=False)
axes[1].scatter(rankdata(X_topo[:, 0])[sample] / n_genes, y[sample], s=6, alpha=0.3, c=gene_proximal[sample], cmap="coolwarm")
axes[1].set_xlabel("Degree (rank-normalized)"); axes[1].set_ylabel("RWR proximity")
axes[1].set_title(f"Monotonic nonlinear: Spearman={spearmanr(X_topo[:,0], y)[0]:.3f}")

axes[2].scatter(pred_gb[sample], y[sample], s=6, alpha=0.3, color="#3b6ea5")
lims = [min(pred_gb.min(), y.min()), max(pred_gb.max(), y.max())]
axes[2].plot(lims, lims, ls="--", color="gray", lw=1)
axes[2].set_xlabel("Predicted RWR (nonlinear, from topology)"); axes[2].set_ylabel("True RWR")
axes[2].set_title(f"Nonlinear recovery: R^2={r2_gb:.3f}")
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "Figure_7_proximity_redundancy.png"), dpi=300, bbox_inches="tight")
plt.close(fig); print("Figure 7 updated.")
gc.collect()

=== PROBE 1b: RWR on RANK-transformed topological features ===
linear R^2 (raw feats) = 0.1027  ->  R^2 (rank feats) = 0.0405  | Spearman(pred,true)=0.9295

=== PROBE 1c: RWR on topology, NONLINEAR (gradient boosting) ===
nonlinear R^2 = 0.0161  | Spearman(pred,true)=0.9025

=== PROBE 2 (recap): GCN embeddings -> RWR ===
linear R^2 = 0.5948  | Spearman(pred,true)=0.5240

=== RECONCILIATION ===
Raw-linear R^2=0.103 (low) but rank-R^2=0.040 and nonlinear-R^2=0.016 (high)
=> RWR is a monotonic NONLINEAR function of centrality; rank/nonlinear models recover it, linear does not.
Step 8b saved.
Figure 7 updated.


8810

# 8c

In [13]:
with open(os.path.join(OUT_DIR, "step8_diagnostic.pkl"), "rb") as f:
    D = pickle.load(f)
with open(os.path.join(OUT_DIR, "step8b_diagnostic.pkl"), "rb") as f:
    Db = pickle.load(f)
with open(os.path.join(OUT_DIR, "step2_graph.pkl"), "rb") as f:
    graph = pickle.load(f)
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, cross_val_predict
from scipy.stats import spearmanr, rankdata

rwr = graph["rwr"]; gene_feat_np = graph["gene_feat"]; n_genes = graph["n_genes"]
gene_proximal = graph["gene_proximal"]
gene_embeddings = D["gene_embeddings"]; feat_names = D["feat_names"]
r2_emb = D["r2_emb"].mean()
X_topo = gene_feat_np[:, :5]
y = rwr.astype(np.float64)

spearmans = [spearmanr(X_topo[:, i], y)[0] for i in range(len(feat_names))]
print("=== RWR vs centrality (Spearman rank correlation) ===")
for n, r in sorted(zip(feat_names, spearmans), key=lambda t: -t[1]):
    print(f"   {n:14s} rho={r:+.4f}")

kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
pred_emb = cross_val_predict(Ridge(alpha=1.0), gene_embeddings, y, cv=kf)
rho_emb = spearmanr(pred_emb, y)[0]
print(f"\nGCN embeddings -> RWR: magnitude R^2={r2_emb:.4f}, rank rho={rho_emb:.4f}")
print(f"RWR skewness: {float(pd.Series(y).skew()):.2f} (heavy right skew => low magnitude-R^2, high rank agreement)")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

order = np.argsort(spearmans)
axes[0].barh([feat_names[i] for i in order], [spearmans[i] for i in order], color="#a53b5a")
axes[0].set_xlabel("Spearman rho with RWR proximity"); axes[0].set_xlim(0, 1)
axes[0].set_title("Disease-proximity is rank-equivalent\nto network centrality")
for i, idx in enumerate(order):
    axes[0].text(spearmans[idx] + 0.01, i, f"{spearmans[idx]:.2f}", va="center", fontsize=9)

sample = np.random.RandomState(SEED).choice(n_genes, min(3000, n_genes), replace=False)
axes[1].scatter(rankdata(X_topo[:, 0])[sample] / n_genes, rankdata(y)[sample] / n_genes,
                s=6, alpha=0.3, c=gene_proximal[sample], cmap="coolwarm")
axes[1].set_xlabel("Degree (rank)"); axes[1].set_ylabel("RWR proximity (rank)")
axes[1].set_title(f"Rank-rank: degree vs RWR (rho={spearmans[0]:.3f})")

axes[2].scatter(pred_emb[sample], y[sample], s=6, alpha=0.3, color="#2a7f3f")
lims = [min(pred_emb.min(), y.min()), max(pred_emb.max(), y.max())]
axes[2].plot(lims, lims, ls="--", color="gray", lw=1)
axes[2].set_xlabel("Predicted RWR (from GCN embeddings)"); axes[2].set_ylabel("True RWR")
axes[2].set_title(f"GCN encodes proximity unsupervised\nR^2={r2_emb:.2f}, rho={rho_emb:.2f}")
ins = axes[2].inset_axes([0.55, 0.55, 0.4, 0.4])
ins.hist(y, bins=50, color="#888"); ins.set_yticks([]); ins.set_title("RWR dist (skewed)", fontsize=7)
ins.tick_params(labelsize=6)

plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "Figure_7_proximity_redundancy.png"), dpi=300, bbox_inches="tight")
plt.close(fig); print("Figure 7 regenerated (rank-led).")

with open(os.path.join(OUT_DIR, "step8c_diagnostic.pkl"), "wb") as f:
    pickle.dump({"spearmans": dict(zip(feat_names, spearmans)), "r2_emb": r2_emb,
                 "rho_emb": rho_emb, "rwr_skew": float(pd.Series(y).skew())}, f)
print("Step 8c saved.")
gc.collect()

=== RWR vs centrality (Spearman rank correlation) ===
   degree         rho=+0.9572
   pagerank       rho=+0.9542
   k-core         rho=+0.9510
   betweenness    rho=+0.9073
   clustering     rho=+0.2582

GCN embeddings -> RWR: magnitude R^2=0.5948, rank rho=0.5240
RWR skewness: 14.51 (heavy right skew => low magnitude-R^2, high rank agreement)
Figure 7 regenerated (rank-led).
Step 8c saved.


8918

# Step 9 — Case study + interpretability + final save

In [14]:
with open(os.path.join(OUT_DIR, "step5_results.pkl"), "rb") as f:
    R5 = pickle.load(f)
with open(os.path.join(OUT_DIR, "step2_graph.pkl"), "rb") as f:
    graph = pickle.load(f)
with open(os.path.join(OUT_DIR, "step1_artifacts.pkl"), "rb") as f:
    A = pickle.load(f)

MODEL_KINDS = R5["MODEL_KINDS"]
n_drugs = graph["n_drugs"]; n_genes = graph["n_genes"]
drug_nodes = graph["drug_nodes"]; drug_idx = graph["drug_idx"]; gene_idx = graph["gene_idx"]
gene_nodes = graph["gene_nodes"]; sym_of_gene = graph["sym_of_gene"]
drug_targets = graph["drug_targets"]; rwr = graph["rwr"]; gene_proximal = graph["gene_proximal"]
seed_in_lcc = graph["seed_in_lcc"]; gg_edge_np = graph["gg_edge"]
ctd_chem = A["ctd_chem"]; pos_drugs_in = graph["pos_drugs_in"]

chem_name = ctd_chem.set_index("chemical_id")["chemical_name"].to_dict()
pos_drug_set = set(drug_idx[d] for d in pos_drugs_in if d in drug_idx)
print("GBM therapeutic drugs in graph (validation positives):", len(pos_drug_set))

prox_genes = np.where(gene_proximal)[0]
prox_genes_t = torch.tensor(prox_genes, dtype=torch.long, device=device)

def rank_drugs(model_state, kind):
    model = build_model(kind)
    model.load_state_dict(model_state); model.eval()
    with torch.no_grad():
        g_emb, d_emb = model.encode(gene_feat, drug_feat, gg_bi, dg_du, dg_gv, gene_prox_t, drug_prox_t)
        gp = g_emb[prox_genes_t]
        w = torch.tensor(rwr[prox_genes], dtype=torch.float32, device=device).unsqueeze(0)
        scores = torch.zeros(n_drugs, device=device)
        B = 2000
        for i in range(0, n_drugs, B):
            de = d_emb[i:i+B]
            s = (de.unsqueeze(1) * model.rel.unsqueeze(0).unsqueeze(0) * gp.unsqueeze(0)).sum(-1)
            scores[i:i+B] = (s * w).sum(1) / w.sum()
        sc = scores.cpu().numpy()
    del model, g_emb, d_emb; gc.collect(); torch.cuda.empty_cache()
    return sc

model_rankings = {}
for kind in MODEL_KINDS:
    sd = torch.load(os.path.join(MODEL_DIR, f"model_{kind}_seed{SEED}.pt"), map_location=device)
    model_rankings[kind] = rank_drugs(sd, kind)
    print(f"{kind}: ranking computed")

def topk_enrichment(scores, pos_set, K_list=[10, 20, 50, 100, 200]):
    order = np.argsort(scores)[::-1]
    rows = []
    N = n_drugs; n_pos = len(pos_set)
    for K in K_list:
        topk = set(order[:K].tolist())
        hits = len(topk & pos_set)
        p_hyper = hypergeom.sf(hits - 1, N, n_pos, K)
        fold = (hits / K) / (n_pos / N) if n_pos > 0 else 0
        rows.append({"K": K, "hits": hits, "expected": K * n_pos / N, "fold": fold, "p_hyper": p_hyper})
    return pd.DataFrame(rows)

print("\n=== TOP-K ENRICHMENT FOR GBM THERAPEUTIC DRUGS (per model) ===")
enrich_all = {}
for kind in MODEL_KINDS:
    df = topk_enrichment(model_rankings[kind], pos_drug_set)
    enrich_all[kind] = df
    print(f"\n{kind}:")
    print(df.to_string(index=False))

ens = np.mean([model_rankings[k] for k in MODEL_KINDS], axis=0)
order = np.argsort(ens)[::-1]
ranked_rows = []
for rank, di_ in enumerate(order, 1):
    d = drug_nodes[di_]
    ranked_rows.append({"rank": rank, "chemical_id": d, "chemical_name": chem_name.get(d, d),
                        "ensemble_score": float(ens[di_]), "is_known_GBM_therapeutic": di_ in pos_drug_set,
                        "n_targets": len(drug_targets.get(d, []))})
ranked_df = pd.DataFrame(ranked_rows)
ranked_df.to_csv(os.path.join(OUT_DIR, "ranked_repurposing_candidates.csv"), index=False)
novel_df = ranked_df[~ranked_df["is_known_GBM_therapeutic"]].reset_index(drop=True)
print("\n=== TOP 20 NOVEL CANDIDATES (ensemble; known GBM therapeutics excluded) ===")
print(novel_df.head(20)[["rank", "chemical_name", "ensemble_score", "n_targets"]].to_string(index=False))
G_ppi = nx.Graph(); G_ppi.add_edges_from(zip(gg_edge_np[0].tolist(), gg_edge_np[1].tolist()))
idx_to_gene = {v: k for k, v in gene_idx.items()}
seed_idx_set = set(gene_idx[g] for g in seed_in_lcc if g in gene_idx)
seed_node_set = set(seed_in_lcc)

def paths_for_drug(d, max_paths=3, cutoff=3):
    tgts = [g for g in drug_targets.get(d, []) if g in gene_idx]
    paths = []
    for t in sorted(tgts, key=lambda g: rwr[gene_idx[g]], reverse=True)[:8]:
        ti = gene_idx[t]
        if ti in seed_idx_set:
            paths.append([("drug", d), ("gene", t)]); continue
        best = None
        for s in sorted(seed_in_lcc, key=lambda g: rwr[gene_idx[g]], reverse=True)[:15]:
            if s in gene_idx and ti in G_ppi and gene_idx[s] in G_ppi:
                try:
                    sp = nx.shortest_path(G_ppi, ti, gene_idx[s])
                    if best is None or len(sp) < len(best): best = sp
                except Exception:
                    pass
        if best is not None and len(best) - 1 <= cutoff:
            paths.append([("drug", d)] + [("gene", idx_to_gene[n]) for n in best])
        if len(paths) >= max_paths: break
    return paths

top_novel = novel_df.head(3)["chemical_id"].tolist()
all_paths = {}
print("\n=== MULTI-HOP PATH RATIONALES (top-3 novel) ===")
for d in top_novel:
    ps = paths_for_drug(d); all_paths[d] = ps
    print(f"\n{chem_name.get(d, d)}:")
    for p in ps:
        s = " -> ".join([chem_name.get(x, x) if t == "drug" else sym_of_gene.get(x, str(x)) for t, x in p])
        print("   " + s + (" [SEED]" if p[-1][1] in seed_node_set else ""))

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
K_list = [10, 20, 50, 100, 200]
w = 0.25
xpos = np.arange(len(K_list))
for i, kind in enumerate(MODEL_KINDS):
    folds = enrich_all[kind]["fold"].values
    axes[0].bar(xpos + i * w, folds, w, label=kind)
    for j, K in enumerate(K_list):
        p = enrich_all[kind]["p_hyper"].values[j]
        if p < 0.05:
            axes[0].text(xpos[j] + i * w, folds[j] + 0.1, "*", ha="center", fontsize=10)
axes[0].axhline(1.0, ls="--", color="gray", lw=1)
axes[0].set_xticks(xpos + w); axes[0].set_xticklabels([f"Top-{k}" for k in K_list])
axes[0].set_ylabel("Fold enrichment"); axes[0].set_title("GBM Therapeutic Enrichment (all models, * p<0.05)")
axes[0].legend(fontsize=8)

for kind in MODEL_KINDS:
    o = np.argsort(model_rankings[kind])[::-1]
    cum = np.cumsum([1 if i in pos_drug_set else 0 for i in o])
    axes[1].plot(np.arange(1, n_drugs + 1), cum, label=kind, lw=1.8)
axes[1].plot([0, n_drugs], [0, len(pos_drug_set)], ls="--", color="gray", label="random")
axes[1].set_xlim(0, 500); axes[1].set_xlabel("Rank"); axes[1].set_ylabel("Cumulative GBM therapeutics")
axes[1].set_title("Cumulative Recovery of Known Therapeutics"); axes[1].legend(fontsize=8)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "Figure_8_enrichment_casestudy.png"), dpi=300, bbox_inches="tight")
plt.close(fig); print("Figure 8 saved.")

fig, ax = plt.subplots(figsize=(12, 8))
H = nx.DiGraph(); node_kind = {}
for d, ps in all_paths.items():
    for p in ps:
        for i in range(len(p) - 1):
            (t1, x1), (t2, x2) = p[i], p[i + 1]
            n1 = chem_name.get(x1, x1) if t1 == "drug" else sym_of_gene.get(x1, str(x1))
            n2 = chem_name.get(x2, x2) if t2 == "drug" else sym_of_gene.get(x2, str(x2))
            H.add_edge(n1, n2); node_kind[n1] = t1; node_kind[n2] = t2
            if t2 == "gene" and x2 in seed_node_set: node_kind[n2] = "seed"
if H.number_of_nodes() > 0:
    pos = nx.spring_layout(H, seed=SEED, k=0.6)
    cmap = {"drug": "#e67e22", "gene": "#3498db", "seed": "#e74c3c"}
    nc = [cmap.get(node_kind.get(n, "gene")) for n in H.nodes()]
    nx.draw_networkx_edges(H, pos, alpha=0.4, arrows=True, arrowsize=12, ax=ax)
    nx.draw_networkx_nodes(H, pos, node_color=nc, node_size=900, ax=ax)
    nx.draw_networkx_labels(H, pos, font_size=7, ax=ax)
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(color=cmap["drug"], label="Drug"), Patch(color=cmap["gene"], label="Gene"), Patch(color=cmap["seed"], label="Disease seed")], loc="upper left")
ax.set_title(f"Interpretability: Multi-Hop Rationales to {DISEASE_NAME} Seeds"); ax.axis("off")
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "Figure_9_interpretability_paths.png"), dpi=300, bbox_inches="tight")
plt.close(fig); print("Figure 9 saved.")

for kind in MODEL_KINDS:
    enrich_all[kind].to_csv(os.path.join(OUT_DIR, f"enrichment_{kind}.csv"), index=False)
torch.save({"model_rankings": model_rankings, "ensemble": ens, "disease": DISEASE_NAME},
           os.path.join(MODEL_DIR, "final_rankings.pt"))
with open(os.path.join(OUT_DIR, "step9_casestudy.pkl"), "wb") as f:
    pickle.dump({"ranked_df": ranked_df, "novel_df": novel_df, "enrich_all": enrich_all, "all_paths": all_paths}, f)
print("\n=== ARTIFACT INVENTORY ===")
for root in [FIG_DIR, MODEL_DIR, OUT_DIR]:
    for fn in sorted(os.listdir(root)):
        fp = os.path.join(root, fn)
        if os.path.isfile(fp):
            print(f"{fp}  ({os.path.getsize(fp)/1024:.0f} KB)")
print("\nStep 9 complete.")
gc.collect(); torch.cuda.empty_cache()

GBM therapeutic drugs in graph (validation positives): 76
gated_gcn: ranking computed
gcn: ranking computed
gat: ranking computed

=== TOP-K ENRICHMENT FOR GBM THERAPEUTIC DRUGS (per model) ===

gated_gcn:
  K  hits  expected      fold      p_hyper
 10     1  0.050993 19.610526 4.985337e-02
 20     1  0.101986  9.805263 9.725247e-02
 50     3  0.254965 11.766316 2.102543e-03
100     6  0.509930 11.766316 1.173294e-05
200    13  1.019860 12.746842 2.252638e-11

gcn:
  K  hits  expected      fold      p_hyper
 10     3  0.050993 58.831579 1.490331e-05
 20     3  0.101986 29.415789 1.364758e-04
 50     5  0.254965 19.610526 5.345105e-06
100     7  0.509930 13.727368 7.412470e-07
200    15  1.019860 14.707895 6.731484e-14

gat:
  K  hits  expected      fold      p_hyper
 10     1  0.050993 19.610526 4.985337e-02
 20     4  0.101986 39.221053 2.843113e-06
 50     7  0.254965 27.454737 5.666023e-09
100     7  0.509930 13.727368 7.412470e-07
200    12  1.019860 11.766316 3.584954e-10

=== TOP

# 9b

In [15]:
with open(os.path.join(OUT_DIR, "step5_results.pkl"), "rb") as f:
    R5 = pickle.load(f)
with open(os.path.join(OUT_DIR, "step2_graph.pkl"), "rb") as f:
    graph = pickle.load(f)
with open(os.path.join(OUT_DIR, "step1_artifacts.pkl"), "rb") as f:
    A = pickle.load(f)
with open(os.path.join(OUT_DIR, "step9_casestudy.pkl"), "rb") as f:
    S9 = pickle.load(f)

MODEL_KINDS = R5["MODEL_KINDS"]
n_drugs = graph["n_drugs"]; drug_nodes = graph["drug_nodes"]; drug_idx = graph["drug_idx"]
gene_idx = graph["gene_idx"]; sym_of_gene = graph["sym_of_gene"]
drug_targets = graph["drug_targets"]; rwr = graph["rwr"]; seed_in_lcc = graph["seed_in_lcc"]
gg_edge_np = graph["gg_edge"]; pos_drugs_in = graph["pos_drugs_in"]
ctd_chem = A["ctd_chem"]; dgidb_mapped = A["dgidb_mapped"]
chem_name = ctd_chem.set_index("chemical_id")["chemical_name"].to_dict()

approved = dgidb_mapped[dgidb_mapped["approved"].astype(str).str.lower() == "true"]
approved_mesh = set(approved["chemical_id"].dropna().str.replace("MESH:", "", regex=False))
antineo = dgidb_mapped[dgidb_mapped["anti_neoplastic"].astype(str).str.lower() == "true"]
antineo_mesh = set(antineo["chemical_id"].dropna().str.replace("MESH:", "", regex=False))
drug_pool_ids = set(d for d in drug_nodes if d in approved_mesh)
drug_pool_mask = np.array([d in drug_pool_ids for d in drug_nodes], dtype=bool)
print("Approved-drug nodes in pool:", int(drug_pool_mask.sum()), "/", n_drugs)
print("(of which anti-neoplastic:", len(set(d for d in drug_pool_ids if d in antineo_mesh)), ")")

pos_drug_set_full = set(drug_idx[d] for d in pos_drugs_in if d in drug_idx)
pos_drug_pool = set(i for i in pos_drug_set_full if drug_pool_mask[i])
print("GBM therapeutic positives within approved pool:", len(pos_drug_pool), "/", len(pos_drug_set_full))

model_rankings = S9 if False else None
import pickle as _p
with open(os.path.join(MODEL_DIR, "final_rankings.pt"), "rb") as f:
    pass
fr = torch.load(os.path.join(MODEL_DIR, "final_rankings.pt"), map_location="cpu")
model_rankings = fr["model_rankings"]

pool_idx = np.where(drug_pool_mask)[0]
def topk_enrichment(scores, pos_set, pool, K_list=[10, 20, 50, 100, 200]):
    pool_sorted = pool[np.argsort(scores[pool])[::-1]]
    N = len(pool); n_pos = len(pos_set & set(pool.tolist()))
    rows = []
    for K in K_list:
        topk = set(pool_sorted[:K].tolist())
        hits = len(topk & pos_set)
        p = hypergeom.sf(hits - 1, N, n_pos, K) if n_pos > 0 else 1.0
        fold = (hits / K) / (n_pos / N) if n_pos > 0 else 0
        rows.append({"K": K, "hits": hits, "expected": K * n_pos / N, "fold": fold, "p_hyper": p})
    return pd.DataFrame(rows)

print("\n=== TOP-K ENRICHMENT (approved-drug pool only) ===")
enrich_all = {}
for kind in MODEL_KINDS:
    df = topk_enrichment(model_rankings[kind], pos_drug_pool, pool_idx)
    enrich_all[kind] = df
    print(f"\n{kind}:\n{df.to_string(index=False)}")

ens = np.mean([model_rankings[k] for k in MODEL_KINDS], axis=0)
pool_order = pool_idx[np.argsort(ens[pool_idx])[::-1]]
ranked_rows = []
for rank, di_ in enumerate(pool_order, 1):
    d = drug_nodes[di_]
    ranked_rows.append({"rank": rank, "chemical_id": d, "chemical_name": chem_name.get(d, d),
                        "ensemble_score": float(ens[di_]), "is_known_GBM_therapeutic": di_ in pos_drug_pool,
                        "anti_neoplastic": d in antineo_mesh, "n_targets": len(drug_targets.get(d, []))})
ranked_df = pd.DataFrame(ranked_rows)
ranked_df.to_csv(os.path.join(OUT_DIR, "ranked_repurposing_candidates_approved.csv"), index=False)
novel_df = ranked_df[~ranked_df["is_known_GBM_therapeutic"]].reset_index(drop=True)
print("\n=== TOP 20 NOVEL APPROVED-DRUG CANDIDATES (ensemble) ===")
print(novel_df.head(20)[["rank", "chemical_name", "ensemble_score", "anti_neoplastic", "n_targets"]].to_string(index=False))

G_ppi = nx.Graph(); G_ppi.add_edges_from(zip(gg_edge_np[0].tolist(), gg_edge_np[1].tolist()))
idx_to_gene = {v: k for k, v in gene_idx.items()}
seed_idx_set = set(gene_idx[g] for g in seed_in_lcc if g in gene_idx); seed_node_set = set(seed_in_lcc)
def paths_for_drug(d, max_paths=3, cutoff=3):
    tgts = [g for g in drug_targets.get(d, []) if g in gene_idx]; paths = []
    for t in sorted(tgts, key=lambda g: rwr[gene_idx[g]], reverse=True)[:8]:
        ti = gene_idx[t]
        if ti in seed_idx_set: paths.append([("drug", d), ("gene", t)]); continue
        best = None
        for s in sorted(seed_in_lcc, key=lambda g: rwr[gene_idx[g]], reverse=True)[:15]:
            if s in gene_idx and ti in G_ppi and gene_idx[s] in G_ppi:
                try:
                    sp = nx.shortest_path(G_ppi, ti, gene_idx[s])
                    if best is None or len(sp) < len(best): best = sp
                except Exception: pass
        if best is not None and len(best) - 1 <= cutoff:
            paths.append([("drug", d)] + [("gene", idx_to_gene[n]) for n in best])
        if len(paths) >= max_paths: break
    return paths
top_novel = novel_df.head(3)["chemical_id"].tolist(); all_paths = {}
print("\n=== PATH RATIONALES (top-3 novel approved drugs) ===")
for d in top_novel:
    ps = paths_for_drug(d); all_paths[d] = ps
    print(f"\n{chem_name.get(d, d)}:")
    for p in ps:
        s = " -> ".join([chem_name.get(x, x) if t == "drug" else sym_of_gene.get(x, str(x)) for t, x in p])
        print("   " + s + (" [SEED]" if p[-1][1] in seed_node_set else ""))

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
K_list = [10, 20, 50, 100, 200]; w = 0.25; xpos = np.arange(len(K_list))
for i, kind in enumerate(MODEL_KINDS):
    folds = enrich_all[kind]["fold"].values
    axes[0].bar(xpos + i * w, folds, w, label=kind)
    for j in range(len(K_list)):
        if enrich_all[kind]["p_hyper"].values[j] < 0.05:
            axes[0].text(xpos[j] + i * w, folds[j] + 0.1, "*", ha="center", fontsize=10)
axes[0].axhline(1.0, ls="--", color="gray", lw=1)
axes[0].set_xticks(xpos + w); axes[0].set_xticklabels([f"Top-{k}" for k in K_list])
axes[0].set_ylabel("Fold enrichment"); axes[0].set_title("GBM Enrichment, Approved-Drug Pool (* p<0.05)"); axes[0].legend(fontsize=8)
for kind in MODEL_KINDS:
    o = pool_idx[np.argsort(model_rankings[kind][pool_idx])[::-1]]
    cum = np.cumsum([1 if i in pos_drug_pool else 0 for i in o])
    axes[1].plot(np.arange(1, len(o) + 1), cum, label=kind, lw=1.8)
axes[1].plot([0, len(pool_idx)], [0, len(pos_drug_pool)], ls="--", color="gray", label="random")
axes[1].set_xlim(0, min(300, len(pool_idx))); axes[1].set_xlabel("Rank (within approved pool)")
axes[1].set_ylabel("Cumulative GBM therapeutics"); axes[1].set_title("Cumulative Recovery"); axes[1].legend(fontsize=8)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "Figure_8_enrichment_casestudy.png"), dpi=300, bbox_inches="tight")
plt.close(fig); print("Figure 8 updated.")

fig, ax = plt.subplots(figsize=(12, 8)); H = nx.DiGraph(); node_kind = {}
for d, ps in all_paths.items():
    for p in ps:
        for i in range(len(p) - 1):
            (t1, x1), (t2, x2) = p[i], p[i + 1]
            n1 = chem_name.get(x1, x1) if t1 == "drug" else sym_of_gene.get(x1, str(x1))
            n2 = chem_name.get(x2, x2) if t2 == "drug" else sym_of_gene.get(x2, str(x2))
            H.add_edge(n1, n2); node_kind[n1] = t1; node_kind[n2] = "seed" if (t2 == "gene" and x2 in seed_node_set) else t2
if H.number_of_nodes() > 0:
    pos = nx.spring_layout(H, seed=SEED, k=0.6); cmap = {"drug": "#e67e22", "gene": "#3498db", "seed": "#e74c3c"}
    nx.draw_networkx_edges(H, pos, alpha=0.4, arrows=True, arrowsize=12, ax=ax)
    nx.draw_networkx_nodes(H, pos, node_color=[cmap.get(node_kind.get(n, "gene")) for n in H.nodes()], node_size=900, ax=ax)
    nx.draw_networkx_labels(H, pos, font_size=7, ax=ax)
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(color=cmap["drug"], label="Drug"), Patch(color=cmap["gene"], label="Gene"), Patch(color=cmap["seed"], label="Disease seed")], loc="upper left")
ax.set_title(f"Interpretability: Multi-Hop Rationales to {DISEASE_NAME} Seeds (approved drugs)"); ax.axis("off")
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "Figure_9_interpretability_paths.png"), dpi=300, bbox_inches="tight")
plt.close(fig); print("Figure 9 updated.")

for kind in MODEL_KINDS:
    enrich_all[kind].to_csv(os.path.join(OUT_DIR, f"enrichment_{kind}_approved.csv"), index=False)
with open(os.path.join(OUT_DIR, "step9b_casestudy.pkl"), "wb") as f:
    pickle.dump({"ranked_df": ranked_df, "novel_df": novel_df, "enrich_all": enrich_all,
                "all_paths": all_paths, "drug_pool_mask": drug_pool_mask, "pos_drug_pool": pos_drug_pool}, f)
print("\nStep 9b complete.")
gc.collect()

Approved-drug nodes in pool: 2138 / 14904
(of which anti-neoplastic: 220 )
GBM therapeutic positives within approved pool: 41 / 76

=== TOP-K ENRICHMENT (approved-drug pool only) ===

gated_gcn:
  K  hits  expected     fold      p_hyper
 10     1  0.191768 5.214634 1.763778e-01
 20     1  0.383536 2.607317 3.222722e-01
 50     8  0.958840 8.343415 2.673308e-06
100    18  1.917680 9.386341 2.066698e-14
200    24  3.835360 6.257561 1.997962e-15

gcn:
  K  hits  expected     fold      p_hyper
 10     1  0.191768 5.214634 1.763778e-01
 20     2  0.383536 5.214634 5.483383e-02
 50     9  0.958840 9.386341 1.979192e-07
100    18  1.917680 9.386341 2.066698e-14
200    26  3.835360 6.779024 6.880014e-18

gat:
  K  hits  expected     fold      p_hyper
 10     1  0.191768 5.214634 1.763778e-01
 20     2  0.383536 5.214634 5.483383e-02
 50     7  0.958840 7.300488 3.050874e-05
100    16  1.917680 8.343415 6.213180e-12
200    27  3.835360 7.039756 3.437667e-19

=== TOP 20 NOVEL APPROVED-DRUG CANDI

13148